[![](imagens/colab-badge.png){width="16%"}](https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/cap03/cap03.EPs_aluno.ipynb)
[![](imagens/github-badge.png){width="20%"}](https://github.com/fzampirolli/pdi-vc)

## 💻 **Parte Prática com Exercícios de Programação**

---

### 🎯 Objetivo deste Caderno {.unnumbered}

O caderno permite desenvolver, validar, organizar e testar soluções de **Exercícios de Programação (EPs)** em ambientes interativos, como o Colab, com os mesmos casos de teste do Moodle, copiando para lá apenas na hora de registrar a nota oficial.

#### *Download* {.unnumbered}

Baixe `morph.py` e `testsuite.py` executando a célula abaixo:

In [1]:
import os, sys, importlib, inspect, urllib.request

BASE_URL = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph"
for f in ["morph.py", "testsuite.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(f"{BASE_URL}/{f}", f)

import morph, testsuite
importlib.reload(morph); importlib.reload(testsuite)
from morph import mm
from testsuite import TestSuite

print(f"\u2705 Ambiente pronto. Morph: {morph.__version__} | TestSuite: {testsuite.__version__}")

✅ Ambiente pronto. Morph: 1.1.0 | TestSuite: 1.1.0


---

#### Executando os Testes {.unnumbered}

Para rodar os testes, execute `TestSuite("EP03_XX.extensão").run()` numa nova célula, trocando a extensão pela da linguagem usada (`.py`, `.java`, `.c`, `.cpp`, `.js` ou `.r`). O sistema baixa os casos de teste do GitHub, executa o programa e calcula a nota automaticamente.

---

### EP03_01 ➕ Adição Saturada de Constante

Em sistemas de vigilância por vídeo, câmeras em ambientes com iluminação variável produzem imagens subexpostas. O ajuste de brilho por **adição saturada de uma constante** é a operação mais simples para correção imediata, sendo aplicada em tempo real nos *chips* de câmeras embarcadas e em *pipelines* de pré-processamento de robôs móveis. 

Ver na @fig-03-sim-adicao uma simulação deste EP.


#### 📋 Diretrizes de Implementação

1. **Dimensões:** Ler os inteiros $L$ (linhas) e $C$ (colunas).
2. **Constante:** Ler o inteiro $k$ (valor a ser somado).
3. **Dados:** Ler os valores inteiros da matriz original linha a linha.
4. **Mapeamento:** Para cada pixel $p$, calcular o novo valor pela equação:

$$p' = \text{clip}(p + k)$$

5. **Saída:** Exibir a matriz resultante com dimensões $L \times C$.

---

#### 📌 Restrições Computacionais

* **Saturação (*Clipping*):** Os valores devem ser confinados ao intervalo $[0, 255]$:
$$\text{clip}(x) = \max(0, \min(255, x))$$
* **Tipo:** O resultado final deve ser inteiro (sem casas decimais).
* **$k$ pode ser negativo:** valores negativos escurecem a imagem; positivos clareiam.

---

#### 🧠 Fundamentação Teórica

| Parâmetro | Tipo | Impacto Visual |
|-----------|------|----------------|
| **$k > 0$** | Inteiro | Clareia a imagem; pixels próximos de 255 saturam em branco |
| **$k < 0$** | Inteiro | Escurece a imagem; pixels próximos de 0 saturam em preto |
| **$k = 0$** | Inteiro | Imagem inalterada |

---

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linha 3: Inteiro $k$.
* Linhas seguintes: Elementos inteiros da matriz original.

**Saída:**

* Matriz transformada em $L$ linhas e $C$ colunas, valores inteiros separados por espaço.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 2<br>3<br>50<br>0 100 200<br>210 240 255 | 50 150 250<br>255 255 255 | Saturação em 255 nos pixels altos |
| 1<br>4<br>-30<br>0 20 200 255 | 0 0 170 225 | Saturação em 0 nos pixels baixos |

In [ ]:
#| label: fig-04-sim-adicao
#| fig-cap: "Simulador: Adição Saturada de Constante"
#| echo: false
#| output: true

from IPython.display import HTML
HTML("""
<div id="sim-ep0401" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Adição Saturada de Constante</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">➕ p' = clip(p + k)</span>
  </div>
  <div style="padding:20px;background:white;">
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:20px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:8px;">
        <label style="font-size:12px;font-weight:bold;color:#27ae60;">k (Constante)</label>
        <span id="ep0401_vl_k" style="font-family:monospace;font-weight:bold;color:#27ae60;">0</span>
      </div>
      <input id="ep0401_sl_k" style="width:100%;accent-color:#27ae60;" max="128" min="-128" step="1" type="range" value="0">
    </div>
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:20px;">
      <div style="text-align:cEP02_01enter;background:white;border:1px solid #eee;padding:15px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:12px;">Entrada Original</p>
        <div id="ep0401_grid_orig" style="display:grid;grid-template-columns:repeat(4,42px);gap:4px;justify-content:center;"></div>
        <button id="ep0401_btn_new" style="margin-top:15px;font-size:11px;padding:5px 10px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">🎲 Nova Imagem</button>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:15px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:12px;">Resultado (p')</p>
        <div id="ep0401_grid_new" style="display:grid;grid-template-columns:repeat(4,42px);gap:4px;justify-content:center;"></div>
        <button id="ep0401_btn_reset" style="margin-top:15px;font-size:11px;padding:5px 10px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">↩ Resetar</button>
      </div>
    </div>
    <div id="ep0401_debug" style="margin-top:20px;background:#e8f5e9;border-radius:8px;padding:10px;border:1px solid #c8e6c9;font-family:monospace;font-size:11px;color:#2e7d32;text-align:center;">Fórmula: clip(p + 0)</div>
  </div>
</div>
<script>
setTimeout(function(){
  var slK=document.getElementById('ep0401_sl_k');
  var vlK=document.getElementById('ep0401_vl_k');
  var gO=document.getElementById('ep0401_grid_orig');
  var gN=document.getElementById('ep0401_grid_new');
  var db=document.getElementById('ep0401_debug');
  if(!slK||!gO)return;
  var pixels=[];
  function generate_0301(){pixels=Array.from({length:16},function(){return Math.floor(Math.random()*256);});}
  function render_0301(){
    var k=parseInt(slK.value);
    vlK.innerText=k;
    db.innerHTML='Fórmula aplicada: <b>clip(p + ('+k+'))</b>';
    gO.innerHTML=''; gN.innerHTML='';
    pixels.forEach(function(p){
      var cO=document.createElement('div');
      cO.style.cssText='width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:bold;border-radius:4px;border:1px solid #eee;';
      cO.style.background='rgb('+p+','+p+','+p+')';
      cO.style.color=p>128?'black':'white';
      cO.innerText=p; gO.appendChild(cO);
      var res=Math.max(0,Math.min(255,p+k));
      var cN=document.createElement('div');
      cN.style.cssText='width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:bold;border-radius:4px;border:1px solid #ddd;';
      cN.style.background='rgb('+res+','+res+','+res+')';
      cN.style.color=res>128?'black':'white';
      cN.innerText=res; gN.appendChild(cN);
    });
  }
  slK.oninput=render_0301;
  document.getElementById('ep0401_btn_new').onclick=function(){generate_0301();render_0301();};
  document.getElementById('ep0401_btn_reset').onclick=function(){slK.value=0;render_0301();};
  generate_0301(); render_0301();
},100);
</script>
""")

In [3]:
%%writefile EP03_01.py
# Código Python

Writing EP03_01.py


In [4]:
TestSuite("EP03_01.py").run()

---

### EP03_02 🔀 Alpha *Blending* de Duas Imagens

Em medicina nuclear, imagens de diferentes modalidades (tomografia computadorizada e ressonância magnética) são fundidas para auxiliar no diagnóstico. A **mistura ponderada** (*alpha blending*) é a operação fundamental desse processo, permitindo ao radiologista controlar interativamente o peso de cada modalidade na imagem exibida.

Ver na @fig-03-sim-blending uma simulação deste EP.


#### 📋 Diretrizes de Implementação

1. **Dimensões:** Ler os inteiros $L$ (linhas) e $C$ (colunas).
2. **Parâmetro:** Ler o valor real $\alpha \in [0, 1]$.
3. **Dados:** Ler os valores inteiros da matriz $f_1$ (imagem 1) e em seguida da matriz $f_2$ (imagem 2).
4. **Mapeamento:** Para cada posição $(i, j)$, calcular:

$$g(i,j) = \text{clip}\left(\text{round}\left(\alpha \cdot f_1(i,j) + (1-\alpha) \cdot f_2(i,j)\right)\right)$$

5. **Saída:** Exibir a matriz resultante $L \times C$.

---

#### 📌 Restrições Computacionais

* **Arredondamento:** Aplicar `round` antes da conversão para inteiro.
* **Saturação:** Confinar ao intervalo $[0, 255]$ com $\text{clip}(x) = \max(0, \min(255, x))$.
* **Operação em float:** Realize a operação em ponto flutuante antes de arredondar.

---

#### 🧠 Fundamentação Teórica

| Valor de $\alpha$ | Resultado |
|:-----------------:|:----------|
| $\alpha = 1.0$ | Apenas $f_1$ |
| $\alpha = 0.5$ | Média aritmética de $f_1$ e $f_2$ |
| $\alpha = 0.0$ | Apenas $f_2$ |

---

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linha 3: Real $\alpha$.
* Linhas seguintes: Elementos de $f_1$ ($L$ linhas com $C$ valores cada).
* Linhas seguintes: Elementos de $f_2$ ($L$ linhas com $C$ valores cada).

**Saída:**

* Matriz resultante $L \times C$.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 1<br>3<br>0.5<br>0 100 200<br>100 200 50 | 50 150 125 | Média entre as duas imagens |
| 1<br>3<br>1.0<br>10 20 30<br>90 80 70 | 10 20 30 | Apenas $f_1$ (alpha=1) |

In [5]:
#| label: fig-03-sim-blending
#| fig-cap: "Simulador: Alpha *Blending* de Duas Imagens"
#| echo: false
#| output: true

from IPython.display import HTML
HTML("""
<div id="sim-ep0302" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Alpha Blending</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🔀 g = α·f1 + (1-α)·f2</span>
  </div>
  <div style="padding:20px;background:white;">
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:20px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:8px;">
        <label style="font-size:12px;font-weight:bold;color:#8e44ad;">α (Alpha)</label>
        <span id="ep0302_vl_a" style="font-family:monospace;font-weight:bold;color:#8e44ad;">0.5</span>
      </div>
      <input id="ep0302_sl_a" style="width:100%;accent-color:#8e44ad;" max="1" min="0" step="0.05" type="range" value="0.5">
    </div>
    <div style="display:grid;grid-template-columns:1fr 1fr 1fr;gap:12px;">
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:8px;">Imagem f1</p>
        <div id="ep0302_grid_f1" style="display:grid;grid-template-columns:repeat(4,38px);gap:3px;justify-content:center;"></div>
        <button id="ep0302_btn_new" style="margin-top:10px;font-size:11px;padding:4px 8px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">🎲 Novas</button>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:8px;">Imagem f2</p>
        <div id="ep0302_grid_f2" style="display:grid;grid-template-columns:repeat(4,38px);gap:3px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:8px;">Resultado g</p>
        <div id="ep0302_grid_g" style="display:grid;grid-template-columns:repeat(4,38px);gap:3px;justify-content:center;"></div>
        <button id="ep0302_btn_reset" style="margin-top:10px;font-size:11px;padding:4px 8px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">↩ Reset</button>
      </div>
    </div>
    <div id="ep0302_debug" style="margin-top:16px;background:#f3e5f5;border-radius:8px;padding:10px;border:1px solid #ce93d8;font-family:monospace;font-size:11px;color:#6a1b9a;text-align:center;">Fórmula: clip(round(0.5 * f1 + 0.5 * f2))</div>
  </div>
</div>
<script>
setTimeout(function(){
  var slA=document.getElementById('ep0302_sl_a');
  var vlA=document.getElementById('ep0302_vl_a');
  var gF1=document.getElementById('ep0302_grid_f1');
  var gF2=document.getElementById('ep0302_grid_f2');
  var gG=document.getElementById('ep0302_grid_g');
  var db=document.getElementById('ep0302_debug');
  if(!slA||!gF1)return;
  var px1=[],px2=[];
  function gen_0302(){px1=Array.from({length:16},function(){return Math.floor(Math.random()*256);});px2=Array.from({length:16},function(){return Math.floor(Math.random()*256);});}
  function cell_0302(v){
    var c=document.createElement('div');
    c.style.cssText='width:38px;height:38px;display:flex;align-items:center;justify-content:center;font-size:9px;font-weight:bold;border-radius:4px;border:1px solid #eee;';
    c.style.background='rgb('+v+','+v+','+v+')';
    c.style.color=v>128?'black':'white';
    c.innerText=v; return c;
  }
  function render_0302(){
    var a=parseFloat(slA.value);
    vlA.innerText=a.toFixed(2);
    db.innerHTML='Fórmula: <b>clip(round('+a.toFixed(2)+' * f1 + '+( 1-a).toFixed(2)+' * f2))</b>';
    gF1.innerHTML=''; gF2.innerHTML=''; gG.innerHTML='';
    for(var i=0;i<16;i++){
      gF1.appendChild(cell_0302(px1[i]));
      gF2.appendChild(cell_0302(px2[i]));
      var r=Math.max(0,Math.min(255,Math.round(a*px1[i]+(1-a)*px2[i])));
      gG.appendChild(cell_0302(r));
    }
  }
  slA.oninput=render_0302;
  document.getElementById('ep0302_btn_new').onclick=function(){gen_0302();render_0302();};
  document.getElementById('ep0302_btn_reset').onclick=function(){slA.value=0.5;render_0302();};
  gen_0302(); render_0302();
},100);
</script>
""")

In [6]:
%%writefile EP03_02.py
# Código Python

Writing EP03_02.py


In [7]:
TestSuite("EP03_02.py").run()

---

### EP03_03 🎭 Inversão de Imagem (Negativo Fotográfico)

Em radiologia, as imagens de raio-X são tradicionalmente visualizadas em negativo: ossos aparecem em preto sobre fundo branco. A operação de **negativo fotográfico** é aplicada rotineiramente em PACS (*Picture Archiving and Communication Systems*) para facilitar a detecção de fraturas e densidades ósseas.

Ver na @fig-03-inversao uma simulação deste EP.


#### 📋 Diretrizes de Implementação

1. **Dimensões:** Ler os inteiros $L$ (linhas) e $C$ (colunas).
2. **Dados:** Ler os valores inteiros da matriz original.
3. **Mapeamento:** Para cada pixel $p$, calcular o negativo:

$$p' = 255 - p$$

4. **Saída:** Exibir a matriz resultante $L \times C$.

---

#### 📌 Restrições Computacionais

* **Sem *clipping* necessário:** O resultado de $255 - p$ com $p \in [0, 255]$ é sempre $\in [0, 255]$.
* **Tipo inteiro:** Saída deve ser valores inteiros.
* **Equivalência lógica:** A operação é idêntica ao `NOT` bit a bit (`mm.bnot`) em imagens de 8 bits.

---

#### 🧠 Fundamentação Teórica

| Pixel Original $p$ | Pixel Negativo $p'$ | Observação |
|:------------------:|:-------------------:|:----------:|
| 0 (preto) | 255 (branco) | Inversão total |
| 128 (cinza médio) | 127 (cinza médio) | Valor central |
| 255 (branco) | 0 (preto) | Inversão total |

---

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linhas seguintes: Elementos inteiros da matriz original.

**Saída:**

* Matriz negativa em $L$ linhas e $C$ colunas.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 1<br>4<br>0 128 200 255 | 255 127 55 0 | Inversão de cada pixel |
| 2<br>2<br>10 20<br>30 40 | 245 235<br>225 215 | Matriz 2x2 invertida |

In [5]:
#| label: fig-03-inversao
#| fig-cap: "Simulador: Inversão de Imagem (Negativo Fotográfico)"
#| echo: false
#| output: true
 
from IPython.display import HTML
HTML("""
<div id="sim-ep0303" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Negativo Fotográfico</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🎭 p' = 255 - p</span>
  </div>
  <div style="padding:20px;background:white;">
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:20px;">
      <div style="text-align:center;background:white;border:1px solid #eee;padding:15px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:12px;">Entrada Original</p>
        <div id="ep0303_grid_orig" style="display:grid;grid-template-columns:repeat(4,42px);gap:4px;justify-content:center;"></div>
        <button id="ep0303_btn_new" style="margin-top:15px;font-size:11px;padding:5px 10px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">🎲 Nova Imagem</button>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:15px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:12px;">Negativo (p'=255-p)</p>
        <div id="ep0303_grid_neg" style="display:grid;grid-template-columns:repeat(4,42px);gap:4px;justify-content:center;"></div>
      </div>
    </div>
    <div id="ep0303_debug" style="margin-top:20px;background:#fce4ec;border-radius:8px;padding:10px;border:1px solid #f48fb1;font-family:monospace;font-size:11px;color:#880e4f;text-align:center;">Fórmula: p' = 255 - p</div>
  </div>
</div>
<script>
setTimeout(function(){
  var gO=document.getElementById('ep0303_grid_orig');
  var gN=document.getElementById('ep0303_grid_neg');
  if(!gO)return;
  var pixels=[];
  function gen_0303(){pixels=Array.from({length:16},function(){return Math.floor(Math.random()*256);});}
  function render_0303(){
    gO.innerHTML=''; gN.innerHTML='';
    pixels.forEach(function(p){
      var cO=document.createElement('div');
      cO.style.cssText='width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:bold;border-radius:4px;border:1px solid #eee;';
      cO.style.background='rgb('+p+','+p+','+p+')';
      cO.style.color=p>128?'black':'white';
      cO.innerText=p; gO.appendChild(cO);
      var r=255-p;
      var cN=document.createElement('div');
      cN.style.cssText='width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:bold;border-radius:4px;border:1px solid #ddd;';
      cN.style.background='rgb('+r+','+r+','+r+')';
      cN.style.color=r>128?'black':'white';
      cN.innerText=r; gN.appendChild(cN);
    });
  }
  document.getElementById('ep0303_btn_new').onclick=function(){gen_0303();render_0303();};
  gen_0303(); render_0303();
},100);
</script>
""")

In [9]:
%%writefile EP03_03.py
# Código Python

Writing EP03_03.py


In [10]:
TestSuite("EP03_03.py").run()

---

### EP03_04 📊 Equalização de Histograma (L bits)

Em imagens de satélite de sensoriamento remoto, a variação de iluminação ao longo do dia produz imagens de baixo contraste. A **equalização de histograma** é aplicada automaticamente em satélites como o Landsat para redistribuir os tons, revelando detalhes de vegetação, relevo e zonas urbanas invisíveis na imagem original.

Ver na @fig-03-equalizacao-simulador uma simulação deste EP.


#### 📋 Diretrizes de Implementação

1. **Dimensões:** Ler os inteiros $L$ (linhas), $C$ (colunas) e $B$ (número de bits, com $L_{\max} = 2^B$).
2. **Dados:** Ler a matriz de pixels $f$ com valores em $[0, 2^B - 1]$.
3. **Histograma:** Calcular $h[k]$ = número de pixels com intensidade $k$, para $k = 0 \ldots 2^B-1$.
4. **Probabilidade:** $p[k] = h[k] / (L \cdot C)$.
5. **CDF:** $\text{cdf}[k] = \sum_{j=0}^{k} p[j]$; função de distribuição acumulada.
6. **LUT:** $\text{lut}[k] = \text{round}\left(\text{cdf}[k] \cdot (2^B - 1)\right)$; *Look-Up Table* (tabela de consulta).
7. **Aplicação:** $g[i,j] = \text{lut}[f[i,j]]$.
8. **Saída:** Exibir a matriz equalizada $L \times C$.

---

#### 📌 Restrições Computacionais

* **Arredondamento:** Usar arredondamento matemático (`round`) na LUT.
* **Bits:** O número de níveis é $2^B$ (ex.: $B=3 \Rightarrow 8$ níveis, $B=8 \Rightarrow 256$ níveis).
* **CDF acumulada:** $\text{cdf}[k] = \sum_{j=0}^{k} p[j]$, com $\text{cdf}[2^B-1] = 1.0$.

---

#### 🧠 Fundamentação Teórica

| Etapa | Operação | Fórmula |
|:-----:|:---------|:--------|
| 1 | Histograma | $h[k] \leftarrow$ nº pixels com intensidade $k$ |
| 2 | Probabilidade | $p[k] = h[k] / (L \cdot C)$ |
| 3 | CDF | $\text{cdf}[k] = \sum_{j=0}^{k} p[j]$ |
| 4 | LUT | $\text{lut}[Look-Up Table (tabela de consulta)k] = \text{round}(\text{cdf}[k] \cdot (2^B-1))$ |
| 5 | Aplicação | $g[i,j] = \text{lut}[f[i,j]]$ |

---

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linha 3: Inteiro $B$ (número de bits).
* Linhas seguintes: Elementos inteiros da matriz.

**Saída:**

* Matriz equalizada em $L$ linhas e $C$ colunas.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 5<br>5<br>3<br>3 4 2 3 4<br>4 3 3 4 3<br>2 3 4 3 2<br>3 4 3 2 3<br>4 3 2 3 4 | 5 7 1 5 7<br>7 5 5 7 5<br>1 5 7 5 1<br>5 7 5 1 5<br>7 5 1 5 7 | Exemplo 3 bits do capítulo |
| 1<br>4<br>3<br>0 0 7 7 | 0 0 7 7 | Histograma bimodal extremo |

In [32]:
#| label: fig-03-equalizacao-simulador
#| fig-cap: "Simulador: Equalização de Histograma (L bits)"
#| echo: false
#| output: true
 
from IPython.display import HTML
HTML("""
<div id="ep0304-sim-eq-03" style="display:block;font-family:sans-serif;line-height:1.4;color:var(--color-text-primary);max-width:900px;margin:24px auto;border:0.5px solid var(--color-border-tertiary);padding:20px;border-radius:var(--border-radius-lg);background:var(--color-background-secondary);">
  <p style="font-size:18px;font-weight:500;margin:0 0 4px;">Simulador: Equalização de Histograma</p>
  <p style="font-size:13px;color:var(--color-text-secondary);margin:0 0 20px;">Escolha o número de bits e gere uma imagem para ver a equalização em tempo real.</p>

  <div style="background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-lg);padding:20px;margin-bottom:20px;">
    <div style="display:flex;justify-content:space-between;margin-bottom:8px;">
      <label style="font-size:13px;font-weight:500;color:#16a085;">B (Bits)</label>
      <span id="ep0304-vl-bits" style="font-family:monospace;font-weight:500;color:#16a085;">3 bits → 8 níveis</span>
    </div>
    <input id="ep0304-sl-bits" style="width:100%;accent-color:#16a085;" max="8" min="1" step="1" type="range" value="3">
    <div style="display:flex;justify-content:space-between;margin-top:6px;font-size:11px;color:var(--color-text-secondary);">
      <span>1 bit (2 níveis)</span><span>4 bits (16 níveis)</span><span>8 bits (256 níveis)</span>
    </div>
  </div>

  <div style="display:grid;grid-template-columns:1fr 1fr;gap:20px;margin-bottom:20px;">
    <div style="background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-lg);padding:15px;text-align:center;">
      <p style="font-size:11px;font-weight:500;color:var(--color-text-secondary);text-transform:uppercase;margin-bottom:12px;">Entrada Original</p>
      <div id="ep0304-grid-orig" style="display:grid;grid-template-columns:repeat(4,42px);gap:4px;justify-content:center;"></div>
      <button id="ep0304-btn-new" style="margin-top:15px;font-size:11px;padding:5px 10px;border-radius:4px;">Nova Imagem</button>
    </div>
    <div style="background:var(--color-background-primary);border:2px solid #16a085;border-radius:var(--border-radius-lg);padding:15px;text-align:center;">
      <p style="font-size:11px;font-weight:500;color:#16a085;text-transform:uppercase;margin-bottom:12px;">Resultado Equalizado</p>
      <div id="ep0304-grid-eq" style="display:grid;grid-template-columns:repeat(4,42px);gap:4px;justify-content:center;"></div>
      <button id="ep0304-btn-reset" style="margin-top:15px;font-size:11px;padding:5px 10px;border-radius:4px;">↩ Resetar</button>
    </div>
  </div>

  <div style="display:grid;grid-template-columns:1fr 1fr;gap:20px;margin-bottom:16px;">
    <div style="background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-lg);padding:15px;">
      <p style="font-size:11px;font-weight:500;color:var(--color-text-secondary);text-transform:uppercase;margin-bottom:10px;text-align:center;">Histograma Original</p>
      <canvas id="ep0304-hist-orig" style="width:100%;height:80px;" width="340" height="80"></canvas>
    </div>
    <div style="background:var(--color-background-primary);border:1px solid #16a085;border-radius:var(--border-radius-lg);padding:15px;">
      <p style="font-size:11px;font-weight:500;color:#16a085;text-transform:uppercase;margin-bottom:10px;text-align:center;">Histograma Equalizado</p>
      <canvas id="ep0304-hist-eq" style="width:100%;height:80px;" width="340" height="80"></canvas>
    </div>
  </div>

  <div style="background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-lg);padding:12px;margin-bottom:16px;overflow-x:auto;">
    <p style="font-size:11px;font-weight:500;color:var(--color-text-secondary);text-transform:uppercase;margin:0 0 8px;">LUT (Tabela de Remapeamento)</p>
    <div id="ep0304-lut-table" style="font-family:monospace;font-size:11px;color:var(--color-text-primary);white-space:nowrap;"></div>
  </div>

  <div id="ep0304-debug-info" style="margin-top:15px;background:#e8f8f5;border-radius:8px;padding:10px;border:1px solid #a3d9cc;font-family:monospace;font-size:11px;color:#0e6655;text-align:center;">
    lut[k] = round( cdf[k] × 7 ) | B=3, níveis=8
  </div>
</div>

<script>
(function ep0304Init() {
  const ep0304Container = document.getElementById('ep0304-sim-eq-03');
  if (!ep0304Container) { setTimeout(ep0304Init, 100); return; }

  const ep0304SlBits      = document.getElementById('ep0304-sl-bits');
  const ep0304VlBits      = document.getElementById('ep0304-vl-bits');
  const ep0304Debug       = document.getElementById('ep0304-debug-info');
  const ep0304GridOrig    = document.getElementById('ep0304-grid-orig');
  const ep0304GridEq      = document.getElementById('ep0304-grid-eq');
  const ep0304LutTable    = document.getElementById('ep0304-lut-table');
  const ep0304HistOrig    = document.getElementById('ep0304-hist-orig');
  const ep0304HistEq      = document.getElementById('ep0304-hist-eq');

  const EP0304_ROWS = 4, EP0304_COLS = 4;
  let ep0304Pixels = [];

  function ep0304GeneratePixels() {
    const ep0304B = parseInt(ep0304SlBits.value);
    const ep0304Levels = Math.pow(2, ep0304B);
    const ep0304Lo = Math.floor(ep0304Levels * 0.2);
    const ep0304Hi = Math.floor(ep0304Levels * 0.5);
    ep0304Pixels = Array.from(
      { length: EP0304_ROWS * EP0304_COLS },
      () => ep0304Lo + Math.floor(Math.random() * (ep0304Hi - ep0304Lo + 1))
    );
  }

  function ep0304ComputeEqualization(ep0304PixArr, ep0304B) {
    const ep0304Levels = Math.pow(2, ep0304B);
    const ep0304N = ep0304PixArr.length;
    const ep0304H = new Array(ep0304Levels).fill(0);
    ep0304PixArr.forEach(p => ep0304H[p]++);
    const ep0304Cdf = new Array(ep0304Levels).fill(0);
    ep0304Cdf[0] = ep0304H[0] / ep0304N;
    for (let k = 1; k < ep0304Levels; k++) ep0304Cdf[k] = ep0304Cdf[k - 1] + ep0304H[k] / ep0304N;
    const ep0304Lut = ep0304Cdf.map(c => Math.round(c * (ep0304Levels - 1)));
    const ep0304Result = ep0304PixArr.map(p => ep0304Lut[p]);
    return { h: ep0304H, cdf: ep0304Cdf, lut: ep0304Lut, result: ep0304Result };
  }

  function ep0304DrawHistogram(ep0304Canvas, ep0304Counts, ep0304Levels, ep0304Color) {
    const ep0304Ctx = ep0304Canvas.getContext('2d');
    const ep0304W = ep0304Canvas.width, ep0304H = ep0304Canvas.height;
    ep0304Ctx.clearRect(0, 0, ep0304W, ep0304H);
    const ep0304MaxVal = Math.max(...ep0304Counts, 1);
    const ep0304BarW = ep0304W / ep0304Levels;
    ep0304Counts.forEach((c, i) => {
      const ep0304BarH = (c / ep0304MaxVal) * (ep0304H - 6);
      ep0304Ctx.fillStyle = ep0304Color;
      ep0304Ctx.fillRect(i * ep0304BarW + 1, ep0304H - ep0304BarH, ep0304BarW - 2, ep0304BarH);
    });
  }

  function ep0304ToGray(ep0304Val, ep0304Levels) {
    return Math.round((ep0304Val / (ep0304Levels - 1)) * 255);
  }

  function ep0304Render() {
    const ep0304B = parseInt(ep0304SlBits.value);
    const ep0304Levels = Math.pow(2, ep0304B);
    ep0304VlBits.innerText = `${ep0304B} bit${ep0304B > 1 ? 's' : ''} → ${ep0304Levels} níveis`;

    const { h: ep0304H, lut: ep0304Lut, result: ep0304Result } = ep0304ComputeEqualization(ep0304Pixels, ep0304B);

    const ep0304HEq = new Array(ep0304Levels).fill(0);
    ep0304Result.forEach(p => ep0304HEq[p]++);

    ep0304LutTable.innerHTML = ep0304Lut.map((v, k) =>
      `<span style="display:inline-block;margin-right:8px;color:var(--color-text-secondary)">${k}→<b style="color:#16a085">${v}</b></span>`
    ).join('');

    ep0304Debug.innerHTML = `<b>lut[k] = round( cdf[k] × ${ep0304Levels - 1} )</b> &nbsp;|&nbsp; B=${ep0304B}, níveis=${ep0304Levels}`;

    ep0304GridOrig.innerHTML = '';
    ep0304GridEq.innerHTML = '';

    ep0304Pixels.forEach((p, i) => {
      const ep0304Gray = ep0304ToGray(p, ep0304Levels);
      const ep0304CellO = document.createElement('div');
      ep0304CellO.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:bold;border-radius:4px;border:1px solid #eee;';
      ep0304CellO.style.background = `rgb(${ep0304Gray},${ep0304Gray},${ep0304Gray})`;
      ep0304CellO.style.color = ep0304Gray > 128 ? '#000' : '#fff';
      ep0304CellO.innerText = p;
      ep0304GridOrig.appendChild(ep0304CellO);

      const ep0304R = ep0304Result[i];
      const ep0304GrayR = ep0304ToGray(ep0304R, ep0304Levels);
      const ep0304CellN = document.createElement('div');
      ep0304CellN.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:bold;border-radius:4px;border:1px solid #ddd;';
      ep0304CellN.style.background = `rgb(${ep0304GrayR},${ep0304GrayR},${ep0304GrayR})`;
      ep0304CellN.style.color = ep0304GrayR > 128 ? '#000' : '#fff';
      ep0304CellN.innerText = ep0304R;
      ep0304GridEq.appendChild(ep0304CellN);
    });

    ep0304DrawHistogram(ep0304HistOrig, ep0304H, ep0304Levels, '#95a5a6');
    ep0304DrawHistogram(ep0304HistEq, ep0304HEq, ep0304Levels, '#16a085');
  }

  ep0304SlBits.oninput = () => { ep0304GeneratePixels(); ep0304Render(); };
  document.getElementById('ep0304-btn-new').onclick = () => { ep0304GeneratePixels(); ep0304Render(); };
  document.getElementById('ep0304-btn-reset').onclick = () => { ep0304GeneratePixels(); ep0304Render(); };

  ep0304GeneratePixels();
  ep0304Render();
})();
</script>

""")

In [12]:
%%writefile EP03_04.py
# Código Python

Writing EP03_04.py


In [13]:
TestSuite("EP03_04.py").run()

---

### EP03_05 🔲 Aplicação de Máscara AND Binária

Em sistemas de inspeção industrial por visão computacional, é necessário isolar regiões de interesse (ROI) em imagens de peças para verificar defeitos de fabricação. A operação **AND bit a bit com uma máscara binária** é o mecanismo fundamental para recortar exatamente a área de inspeção, zerando todos os pixels fora dela.

Ver na @fig-03-mascara-and uma simulação deste EP.


#### 📋 Diretrizes de Implementação

1. **Dimensões:** Ler os inteiros $L$ (linhas) e $C$ (colunas).
2. **Dados:** Ler a matriz de pixels $f$ (valores $\in [0, 255]$).
3. **Máscara:** Ler a matriz binária $m$ (valores: apenas 0 ou 255).
4. **Mapeamento:** Para cada pixel $(i,j)$, aplicar o AND bit a bit:

$$
g(i,j) = f(i,j) \;\text{AND}\; m(i,j)
$$

onde $255 =$ `11111111` e $0 =$ `00000000` em binário.

5. **Saída:** Exibir a matriz resultante $L \times C$.

---

#### 📌 Restrições Computacionais

* **AND com 255:** $p \; \text{AND} \; 255 = p$ (todos os bits preservados).
* **AND com 0:** $p \; \text{AND} \; 0 = 0$ (todos os bits zerados).
* **Máscara:** Os únicos valores possíveis na máscara são 0 e 255.
* **Implementação:** Em Python, o AND bit a bit entre inteiros usa o operador `&`.

---

#### 🧠 Fundamentação Teórica

| Pixel $f$ | Máscara $m$ | Resultado $f$ AND $m$ |
|:---------:|:-----------:|:---------------------:|
| qualquer $v$ | 255 (`11111111`) | $v$ (preservado) |
| qualquer $v$ | 0 (`00000000`) | 0 (zerado) |

---

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linhas seguintes: Elementos de $f$ ($L$ linhas).
* Linhas seguintes: Elementos de $m$ ($L$ linhas com valores 0 ou 255).

**Saída:**

* Matriz resultante $L \times C$.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 2<br>3<br>100 150 200<br>50 80 120<br>255 255 0<br>0 255 255 | 100 150 0<br>0 80 120 | Máscara seleciona região |
| 1<br>4<br>10 20 30 40<br>255 0 255 0 | 10 0 30 0 | Alternado preservado/zerado |

In [1]:
#| label: fig-03-mascara-and
#| fig-cap: "Simulador: Aplicação de Máscara AND Binária"
#| echo: false
#| output: true

from IPython.display import HTML
HTML("""
<style>
.ep0305-cell{width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:11px;font-weight:500;border-radius:6px;transition:transform 0.1s;}
.ep0305-mask-cell{cursor:pointer;border:2px solid;}
.ep0305-mask-cell:hover{transform:scale(1.08);}
.ep0305-mask-cell:active{transform:scale(0.95);}
.ep0305-card{background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-lg);padding:16px;text-align:center;}
.ep0305-label{font-size:11px;font-weight:500;color:var(--color-text-secondary);text-transform:uppercase;letter-spacing:.05em;margin:0 0 4px;}
.ep0305-formula{font-family:var(--font-mono);font-size:13px;color:var(--color-text-primary);margin:0;}
.ep0305-grid{display:grid;grid-template-columns:repeat(4,42px);gap:4px;justify-content:center;margin-top:12px;}
.ep0305-stat{background:var(--color-background-secondary);border-radius:var(--border-radius-md);padding:8px 14px;font-size:12px;color:var(--color-text-secondary);}
.ep0305-stat b{font-weight:500;font-size:15px;display:block;color:var(--color-text-primary);}
</style>

<h2 class="sr-only">Simulador de máscara AND binária: aplica operação AND pixel a pixel entre imagem e máscara interativa.</h2>

<div id="ep0305-sim-and-03" style="max-width:900px;margin:0 auto;padding:4px 0 16px;">

  <div style="display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:14px;margin-bottom:16px;">

    <div class="ep0305-card">
      <p class="ep0305-label">Imagem f</p>
      <p class="ep0305-formula">valores 0–255</p>
      <div id="ep0305-grid-f" class="ep0305-grid"></div>
      <button id="ep0305-btn-new" style="margin-top:14px;font-size:11px;padding:5px 12px;">
        <i class="ti ti-dice" aria-hidden="true"></i> Nova imagem
      </button>
    </div>

    <div class="ep0305-card" style="border:2px solid #185FA5;">
      <p class="ep0305-label" style="color:#185FA5;">Máscara m</p>
      <p class="ep0305-formula" style="color:#185FA5;">clique para alternar</p>
      <div id="ep0305-grid-mask" class="ep0305-grid"></div>
      <button id="ep0305-btn-reset" style="margin-top:14px;font-size:11px;padding:5px 12px;">
        <i class="ti ti-refresh" aria-hidden="true"></i> Resetar máscara
      </button>
    </div>

    <div class="ep0305-card" style="border:2px solid #0F6E56;">
      <p class="ep0305-label" style="color:#0F6E56;">Resultado g</p>
      <p class="ep0305-formula" style="color:#0F6E56;">g = f AND m</p>
      <div id="ep0305-grid-result" class="ep0305-grid"></div>
      <div style="margin-top:14px;height:26px;display:flex;align-items:center;justify-content:center;">
        <span id="ep0305-pct" style="font-size:12px;color:var(--color-text-secondary);font-family:var(--font-mono);"></span>
      </div>
    </div>
  </div>

  <div style="display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:10px;margin-bottom:16px;">
    <div class="ep0305-stat"><b id="ep0305-stat-preserved">—</b>preservados</div>
    <div class="ep0305-stat"><b id="ep0305-stat-zeroed">—</b>zerados</div>
    <div class="ep0305-stat"><b id="ep0305-stat-ratio">—</b>% da imagem visível</div>
  </div>

  <div style="background:var(--color-background-secondary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-lg);padding:14px;display:flex;gap:20px;align-items:center;flex-wrap:wrap;justify-content:center;margin-bottom:14px;">
    <span style="font-size:12px;font-weight:500;color:var(--color-text-secondary);">Legenda da máscara</span>
    <div style="display:flex;align-items:center;gap:8px;">
      <div style="width:32px;height:32px;background:#E6F1FB;border:2px solid #185FA5;border-radius:6px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:500;color:#0C447C;">255</div>
      <span style="font-size:12px;color:var(--color-text-secondary);">Passante — pixel preservado</span>
    </div>
    <div style="display:flex;align-items:center;gap:8px;">
      <div style="width:32px;height:32px;background:#2C2C2A;border:2px solid #444441;border-radius:6px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:500;color:#B4B2A9;">0</div>
      <span style="font-size:12px;color:var(--color-text-secondary);">Bloqueante — pixel zerado</span>
    </div>
  </div>

  <div style="background:#E6F1FB;border:0.5px solid #B5D4F4;border-radius:var(--border-radius-md);padding:10px 14px;font-family:var(--font-mono);font-size:12px;color:#0C447C;text-align:center;">
    <span id="ep0305-debug">g(i,j) = f(i,j) &amp; m(i,j)</span>
  </div>
</div>

<script>
(function ep0305Init() {
  const ep0305Container = document.getElementById('ep0305-sim-and-03');
  if (!ep0305Container) { setTimeout(ep0305Init, 100); return; }

  const ep0305GridF      = document.getElementById('ep0305-grid-f');
  const ep0305GridMask   = document.getElementById('ep0305-grid-mask');
  const ep0305GridResult = document.getElementById('ep0305-grid-result');
  const ep0305Debug      = document.getElementById('ep0305-debug');
  const ep0305Pct        = document.getElementById('ep0305-pct');
  const ep0305StatPres   = document.getElementById('ep0305-stat-preserved');
  const ep0305StatZero   = document.getElementById('ep0305-stat-zeroed');
  const ep0305StatRatio  = document.getElementById('ep0305-stat-ratio');

  const EP0305_N = 16;
  let ep0305Pixels = [];
  let ep0305Mask   = [];

  function ep0305GeneratePixels() {
    ep0305Pixels = Array.from({ length: EP0305_N }, () => Math.floor(Math.random() * 256));
  }

  function ep0305ResetMask() {
    ep0305Mask = Array.from({ length: EP0305_N }, (_, i) => {
      const r = Math.floor(i / 4), c = i % 4;
      return (r + c) % 2 === 0 ? 255 : 0;
    });
  }

  function ep0305TextColor(gray) {
    return gray > 140 ? '#111' : '#eee';
  }

  function ep0305Render() {
    ep0305GridF.innerHTML = '';
    ep0305GridMask.innerHTML = '';
    ep0305GridResult.innerHTML = '';

    let ep0305Preserved = 0, ep0305Zeroed = 0;

    for (let i = 0; i < EP0305_N; i++) {
      const p = ep0305Pixels[i];
      const m = ep0305Mask[i];
      const res = p & m;

      const ep0305CellF = document.createElement('div');
      ep0305CellF.className = 'ep0305-cell';
      ep0305CellF.style.background = `rgb(${p},${p},${p})`;
      ep0305CellF.style.color = ep0305TextColor(p);
      ep0305CellF.style.border = '0.5px solid rgba(128,128,128,0.2)';
      ep0305CellF.innerText = p;
      ep0305GridF.appendChild(ep0305CellF);

      const ep0305CellM = document.createElement('div');
      ep0305CellM.className = 'ep0305-cell ep0305-mask-cell';
      if (m === 255) {
        ep0305CellM.style.background = '#E6F1FB';
        ep0305CellM.style.borderColor = '#185FA5';
        ep0305CellM.style.color = '#0C447C';
      } else {
        ep0305CellM.style.background = '#2C2C2A';
        ep0305CellM.style.borderColor = '#5F5E5A';
        ep0305CellM.style.color = '#B4B2A9';
      }
      ep0305CellM.innerText = m;
      ep0305CellM.setAttribute('title', m === 255 ? 'Clique para bloquear' : 'Clique para passar');
      const ep0305Idx = i;
      ep0305CellM.onclick = () => {
        ep0305Mask[ep0305Idx] = ep0305Mask[ep0305Idx] === 255 ? 0 : 255;
        ep0305Render();
      };
      ep0305GridMask.appendChild(ep0305CellM);

      const ep0305CellR = document.createElement('div');
      ep0305CellR.className = 'ep0305-cell';
      ep0305CellR.style.background = `rgb(${res},${res},${res})`;
      ep0305CellR.style.color = ep0305TextColor(res);
      ep0305CellR.style.border = m === 255
        ? '1.5px solid #1D9E75'
        : '0.5px solid rgba(128,128,128,0.15)';
      ep0305CellR.innerText = res;
      ep0305GridResult.appendChild(ep0305CellR);

      if (m === 255) ep0305Preserved++; else ep0305Zeroed++;
    }

    const ep0305RatioPct = Math.round((ep0305Preserved / EP0305_N) * 100);
    ep0305StatPres.innerText = ep0305Preserved;
    ep0305StatZero.innerText = ep0305Zeroed;
    ep0305StatRatio.innerText = ep0305RatioPct + '%';
    ep0305Pct.innerText = ep0305RatioPct + '% visível';
    ep0305Debug.innerHTML =
      `g(i,j) = f(i,j) &amp; m(i,j) &nbsp;|&nbsp; ${ep0305Preserved} preservados · ${ep0305Zeroed} zerados`;
  }

  document.getElementById('ep0305-btn-new').onclick = () => { ep0305GeneratePixels(); ep0305Render(); };
  document.getElementById('ep0305-btn-reset').onclick = () => { ep0305ResetMask(); ep0305Render(); };

  ep0305GeneratePixels();
  ep0305ResetMask();
  ep0305Render();
})();
</script>

""")

In [15]:
%%writefile EP03_05.py
# Código Python

Writing EP03_05.py


In [16]:
TestSuite("EP03_05.py").run()

---

### EP03_06 🌫️ Filtro de Média com *Kernel* N×N


Em câmeras de veículos autônomos, imagens capturadas sob chuva ou névoa apresentam ruído gaussiano. O **filtro de média** é amplamente utilizado para sua redução em tempo real, sendo implementado diretamente no **ISP** (*Image Signal Processor*) de sensores **CMOS** (*Complementary Metal-Oxide-Semiconductor*).

Os sensores CMOS são os sensores de imagem usados na maioria das câmeras modernas (*smartphones*, *webcams*, câmeras automotivas etc.). Eles convertem a luz em sinais elétricos, e o ISP processa esses sinais em tempo real — aplicando operações como redução de ruído, balanço de branco e outros ajustes de imagem.

Ver na @fig-03-filtro-media-simulador uma simulação deste EP.


#### 📋 Diretrizes de Implementação

1. **Dimensões:** Ler os inteiros $L$ (linhas), $C$ (colunas) e $N$ (tamanho do *kernel*, sempre ímpar).
2. **Dados:** Ler a matriz de pixels $f$.
3. **Filtro de Média:** Para cada pixel $(i,j)$ **interno** (sem bordas), calcular:

$$g(i,j) = \text{round}\left(\frac{1}{N^2} \sum_{s=-(r)}^{r} \sum_{t=-(r)}^{r} f(i+s,\, j+t)\right), \quad r = \lfloor N/2 \rfloor$$

4. **Tratamento de Borda:** Pixels na borda (onde a janela $N \times N$ ultrapassa os limites) devem ser **copiados diretamente** do original sem modificação.
5. **Saída:** Exibir a matriz resultante $L \times C$.

---

#### 📌 Restrições Computacionais

* **Raio:** $r = \lfloor N/2 \rfloor$ (metade do *kernel*, inteiro).
* **Pixels internos:** $(i,j)$ com $r \le i < L-r$ e $r \le j < C-r$.
* **Arredondamento:** Usar arredondamento matemático antes de converter para inteiro.
* **Sem *clipping*:** A média de valores $\in [0,255]$ permanece em $[0,255]$.

---

#### 🧠 Fundamentação Teórica

| Tamanho $N$ | Coeficiente | Pixels na janela | Efeito |
|:-----------:|:-----------:|:----------------:|:------:|
| 3 | $1/9 \approx 0.111$ | 9 | Suave |
| 5 | $1/25 = 0.04$ | 25 | Médio |
| 7 | $1/49 \approx 0.020$ | 49 | Forte |

---

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linha 3: Inteiro $N$ (ímpar, $N \ge 3$).
* Linhas seguintes: Elementos da matriz original.

**Saída:**

* Matriz filtrada $L \times C$.

#### 📌 Exemplos
| Entrada | Saída | Observação |
|---------|-------|------------|
| 3<br>3<br>3<br>10 20 30<br>40 50 60<br>70 80 90 | 10 20 30<br>40 50 60<br>70 80 90 | Apenas borda (3×3 = borda total) |
| 5<br>5<br>3<br>0 0 0 0 0<br>0 0 0 0 0<br>0 0 100 0 0<br>0 0 0 0 0<br>0 0 0 0 0 | 0 0 0 0 0<br>0 11 11 11 0<br>0 11 11 11 0<br>0 11 11 11 0<br>0 0 0 0 0 | Pixel isolado: todos os 9 pixels internos cuja janela 3×3 inclui o valor 100 recebem round(100/9)=11 |

In [3]:
#| label: fig-03-filtro-media-simulador
#| fig-cap: "Simulador: Filtro de Média com *Kernel* N×N"
#| echo: false
#| output: true

from IPython.display import HTML
HTML("""
<style>
.ep0306-cell{width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:500;border-radius:6px;transition:background 0.1s,transform 0.1s,box-shadow 0.1s;}
.ep0306-card{background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-lg);padding:16px;text-align:center;}
.ep0306-label{font-size:11px;font-weight:500;color:var(--color-text-secondary);text-transform:uppercase;letter-spacing:.05em;margin:0 0 4px;}
.ep0306-grid{display:grid;gap:4px;justify-content:center;margin-top:12px;}
.ep0306-stat{background:var(--color-background-secondary);border-radius:var(--border-radius-md);padding:8px 14px;font-size:12px;color:var(--color-text-secondary);text-align:center;}
.ep0306-stat b{font-weight:500;font-size:15px;display:block;color:var(--color-text-primary);}
.ep0306-kbtn{padding:6px 16px;font-size:12px;border-radius:var(--border-radius-md);cursor:pointer;border:2px solid var(--color-border-secondary);background:var(--color-background-primary);color:var(--color-text-secondary);transition:all 0.15s;}
.ep0306-kbtn.active{background:#E6F1FB;border-color:#185FA5;color:#0C447C;font-weight:500;}
</style>

<h2 class="sr-only">Simulador de filtro de média com kernel 3×3 ou 5×5: passe o mouse nas células do resultado para visualizar a janela de vizinhança.</h2>

<div id="ep0306-sim-media-03" style="max-width:900px;margin:0 auto;padding:4px 0 16px;">

  <div style="display:flex;align-items:center;gap:10px;margin-bottom:16px;flex-wrap:wrap;">
    <span style="font-size:13px;font-weight:500;color:var(--color-text-secondary);">Tamanho do kernel:</span>
    <button class="ep0306-kbtn active" id="ep0306-btn-k3" onclick="ep0306SetKernel(3)">3 × 3 &nbsp;<span style="font-weight:400;font-size:11px;">(9 vizinhos)</span></button>
    <button class="ep0306-kbtn" id="ep0306-btn-k5" onclick="ep0306SetKernel(5)">5 × 5 &nbsp;<span style="font-weight:400;font-size:11px;">(25 vizinhos)</span></button>
    <button id="ep0306-btn-new" style="margin-left:auto;font-size:12px;padding:6px 14px;">
      <i class="ti ti-dice" aria-hidden="true"></i> Nova imagem
    </button>
  </div>

  <div style="display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:14px;margin-bottom:14px;">
    <div class="ep0306-card">
      <p class="ep0306-label">Imagem original f (7×7)</p>
      <p style="font-size:11px;color:var(--color-text-secondary);margin:0;">com ruído sal e pimenta</p>
      <div id="ep0306-grid-f" class="ep0306-grid"></div>
    </div>
    <div class="ep0306-card" style="border:2px solid #185FA5;">
      <p class="ep0306-label" style="color:#185FA5;">Resultado g (filtro suavizado)</p>
      <p style="font-size:11px;color:#185FA5;margin:0;">passe o mouse para inspecionar</p>
      <div id="ep0306-grid-result" class="ep0306-grid"></div>
    </div>
  </div>

  <div style="background:var(--color-background-secondary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-lg);padding:12px;display:flex;gap:20px;align-items:center;flex-wrap:wrap;justify-content:center;margin-bottom:12px;">
    <span style="font-size:12px;font-weight:500;color:var(--color-text-secondary);">Legenda</span>
    <div style="display:flex;align-items:center;gap:8px;">
      <div style="width:20px;height:20px;background:#FAEEDA;border:2px dashed #BA7517;border-radius:4px;"></div>
      <span style="font-size:12px;color:var(--color-text-secondary);">Janela do kernel (vizinhos)</span>
    </div>
    <div style="display:flex;align-items:center;gap:8px;">
      <div style="width:20px;height:20px;background:#E1F5EE;border:1.5px solid #1D9E75;border-radius:4px;"></div>
      <span style="font-size:12px;color:var(--color-text-secondary);">Borda — copiada sem filtro</span>
    </div>
    <div style="display:flex;align-items:center;gap:8px;">
      <div style="width:20px;height:20px;background:#E6F1FB;border:1.5px solid #185FA5;border-radius:4px;"></div>
      <span style="font-size:12px;color:var(--color-text-secondary);">Pixel inspecionado</span>
    </div>
  </div>

  <div id="ep0306-debug" style="background:#E6F1FB;border:0.5px solid #B5D4F4;border-radius:var(--border-radius-md);padding:10px 14px;font-family:var(--font-mono);font-size:12px;color:#0C447C;text-align:center;min-height:38px;line-height:1.6;">
    Passe o mouse sobre um pixel interno do resultado para ver o cálculo.
  </div>
</div>

<script>
(function(){
  const EP0306_ROWS = 7, EP0306_COLS = 7, EP0306_N = 49;
  let ep0306Pixels = [], ep0306Result = [], ep0306KSize = 3, ep0306Radius = 1;

  function ep0306GenPixels() {
    ep0306Pixels = Array.from({ length: EP0306_N }, () => {
      let v = 60 + Math.floor(Math.random() * 60);
      if (Math.random() > 0.82) v = Math.random() > 0.5 ? 255 : 0;
      return v;
    });
  }

  function ep0306CalcFilter() {
    ep0306Result = [...ep0306Pixels];
    ep0306Radius = Math.floor(ep0306KSize / 2);
    for (let r = 0; r < EP0306_ROWS; r++) {
      for (let c = 0; c < EP0306_COLS; c++) {
        if (r >= ep0306Radius && r < EP0306_ROWS - ep0306Radius && c >= ep0306Radius && c < EP0306_COLS - ep0306Radius) {
          let sum = 0, cnt = 0;
          for (let s = -ep0306Radius; s <= ep0306Radius; s++)
            for (let t = -ep0306Radius; t <= ep0306Radius; t++) { sum += ep0306Pixels[(r+s)*EP0306_COLS+(c+t)]; cnt++; }
          ep0306Result[r*EP0306_COLS+c] = Math.round(sum/cnt);
        }
      }
    }
  }

  function ep0306TC(g){ return g > 140 ? '#111' : '#eee'; }

  function ep0306HighlightKernel(tr, tc, on) {
    const cells = document.getElementById('ep0306-grid-f').children;
    for (let r = 0; r < EP0306_ROWS; r++) {
      for (let c = 0; c < EP0306_COLS; c++) {
        const idx = r*EP0306_COLS+c;
        if (!cells[idx]) continue;
        const p = ep0306Pixels[idx];
        if (on && Math.abs(r-tr) <= ep0306Radius && Math.abs(c-tc) <= ep0306Radius) {
          cells[idx].style.background = '#FAEEDA';
          cells[idx].style.color = '#633806';
          cells[idx].style.boxShadow = '0 0 0 2px #BA7517 inset';
        } else {
          cells[idx].style.background = `rgb(${p},${p},${p})`;
          cells[idx].style.color = ep0306TC(p);
          cells[idx].style.boxShadow = 'none';
        }
      }
    }
  }

  function ep0306Render() {
    const gf = document.getElementById('ep0306-grid-f');
    const gr = document.getElementById('ep0306-grid-result');
    const cols = `repeat(${EP0306_COLS}, 42px)`;
    gf.style.gridTemplateColumns = cols;
    gr.style.gridTemplateColumns = cols;
    gf.innerHTML = ''; gr.innerHTML = '';

    for (let i = 0; i < EP0306_N; i++) {
      const r = Math.floor(i/EP0306_COLS), c = i%EP0306_COLS;
      const p = ep0306Pixels[i], res = ep0306Result[i];
      const isBorder = r < ep0306Radius || r >= EP0306_ROWS-ep0306Radius || c < ep0306Radius || c >= EP0306_COLS-ep0306Radius;

      const cf = document.createElement('div');
      cf.className = 'ep0306-cell';
      cf.style.background = `rgb(${p},${p},${p})`;
      cf.style.color = ep0306TC(p);
      cf.style.border = '0.5px solid rgba(128,128,128,0.15)';
      cf.innerText = p;
      gf.appendChild(cf);

      const cr = document.createElement('div');
      cr.className = 'ep0306-cell';
      cr.style.background = `rgb(${res},${res},${res})`;
      cr.style.color = ep0306TC(res);
      cr.innerText = res;

      if (isBorder) {
        cr.style.border = '1.5px solid #1D9E75';
        cr.style.background = '#E1F5EE';
        cr.style.color = '#085041';
        cr.onmouseenter = () => {
          document.getElementById('ep0306-debug').innerHTML =
            `Pixel (${r},${c}) é <b>borda</b>: valor herdado do original sem cálculo &nbsp;→&nbsp; <b>${res}</b>`;
        };
        cr.onmouseleave = () => ep0306ResetDebug();
      } else {
        cr.style.border = '0.5px solid var(--color-border-tertiary)';
        cr.style.cursor = 'pointer';
        cr.onmouseenter = () => {
          ep0306HighlightKernel(r, c, true);
          cr.style.transform = 'scale(1.12)';
          cr.style.boxShadow = '0 0 0 2px #185FA5 inset';
          cr.style.background = '#E6F1FB';
          cr.style.color = '#0C447C';
          const neighbors = [];
          for (let s = -ep0306Radius; s <= ep0306Radius; s++)
            for (let t = -ep0306Radius; t <= ep0306Radius; t++)
              neighbors.push(ep0306Pixels[(r+s)*EP0306_COLS+(c+t)]);
          const k = ep0306KSize * ep0306KSize;
          document.getElementById('ep0306-debug').innerHTML =
            `Pixel (${r},${c}): round( (${neighbors.join(' + ')}) / ${k} ) &nbsp;=&nbsp; <b>${res}</b>`;
        };
        cr.onmouseleave = () => {
          ep0306HighlightKernel(r, c, false);
          cr.style.transform = 'scale(1)';
          cr.style.boxShadow = 'none';
          cr.style.background = `rgb(${res},${res},${res})`;
          cr.style.color = ep0306TC(res);
          ep0306ResetDebug();
        };
      }
      gr.appendChild(cr);
    }
  }

  function ep0306ResetDebug() {
    document.getElementById('ep0306-debug').innerText =
      'Passe o mouse sobre um pixel interno do resultado para ver o cálculo.';
  }

  window.ep0306SetKernel = function(k) {
    ep0306KSize = k;
    document.getElementById('ep0306-btn-k3').classList.toggle('active', k===3);
    document.getElementById('ep0306-btn-k5').classList.toggle('active', k===5);
    ep0306CalcFilter();
    ep0306Render();
  };

  document.getElementById('ep0306-btn-new').onclick = () => {
    ep0306GenPixels(); ep0306CalcFilter(); ep0306Render();
  };

  ep0306GenPixels(); ep0306CalcFilter(); ep0306Render();
})();
</script>

""")

In [18]:
%%writefile EP03_06.py
# Código Python

Writing EP03_06.py


In [19]:
TestSuite("EP03_06.py").run()

---

### EP03_07 🔍 Operador Laplaciano (w4) para Realce de Bordas

Em tomografias de alta resolução, a nitidez das bordas entre tecidos é crítica para diagnóstico. O **operador Laplaciano** é amplamente utilizado em *pipelines* de pré-processamento de imagens médicas para realçar automaticamente os contornos anatômicos antes da segmentação, evitando intervenção manual do radiologista.

Ver na @fig-03-laplaciano-simulador uma simulação deste EP.


#### 📋 Diretrizes de Implementação

1. **Dimensões:** Ler os inteiros $L$ (linhas) e $C$ (colunas).
2. **Dados:** Ler a matriz de pixels $f$.
3. **Laplaciano (w4):** Para cada pixel **interno** $(i,j)$ com $1 \le i < L-1$, $1 \le j < C-1$, calcular:

$$\nabla^2 f(i,j) = f(i-1,j) + f(i+1,j) + f(i,j-1) + f(i,j+1) - 4 \cdot f(i,j)$$

4. **Realce:** Calcular a imagem realçada:

$$g(i,j) = \text{clip}(f(i,j) - \nabla^2 f(i,j))$$

5. **Borda:** Pixels na borda são copiados diretamente: $g(i,j) = f(i,j)$.
6. **Saída:** Exibir a matriz realçada $L \times C$.

---

#### 📌 Restrições Computacionais

* ***Kernel* w4:** $\begin{bmatrix} 0 & 1 & 0 \\ 1 & -4 & 1 \\ 0 & 1 & 0 \end{bmatrix}$ — apenas 4-vizinhos.
* **Saturação:** $\text{clip}(x) = \max(0, \min(255, x))$ aplicado ao resultado do realce.
* **Sem arredondamento:** O Laplaciano usa apenas somas/subtrações de inteiros.

---

#### 🧠 Fundamentação Teórica

| Região | $\nabla^2 f$ | Efeito do Realce |
|:------:|:------------:|:----------------:|
| Uniforme | $\approx 0$ | Sem alteração |
| Borda crescente | $< 0$ | Pixel clareado |
| Borda decrescente | $> 0$ | Pixel escurecido |

---

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linhas seguintes: Elementos da matriz original.

**Saída:**

* Matriz realçada $L \times C$.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 3<br>3<br>0 0 0<br>0 100 0<br>0 0 0 | 0 0 0<br>0 255 0<br>0 0 0 | Pico isolado: lap=−400, g=100−(−400)=500 → clip=255 |
| 3<br>3<br>50 50 50<br>50 50 50<br>50 50 50 | 50 50 50<br>50 50 50<br>50 50 50 | Região uniforme: Laplaciano=0, sem alteração |

In [7]:
#| label: fig-03-laplaciano-simulador
#| fig-cap: "Simulador: Operador Laplaciano (w4) para Realce de Bordas"
#| echo: false
#| output: true

from IPython.display import HTML
HTML("""


<style>
.lp-cell{width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9px;font-weight:500;border-radius:4px;transition:background 0.1s,transform 0.12s,box-shadow 0.1s;}
.lp-card{background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-lg);padding:12px 14px;text-align:center;}
.lp-label{font-size:11px;font-weight:500;text-transform:uppercase;letter-spacing:.05em;margin:0 0 2px;}
.lp-sublabel{font-size:10px;margin:0 0 8px;}
.lp-grid{display:grid;grid-template-columns:repeat(5,36px);gap:3px;justify-content:center;margin-top:6px;}
.lp-kbtn{padding:5px 13px;font-size:12px;border-radius:var(--border-radius-md);cursor:pointer;border:1.5px solid var(--color-border-secondary);background:var(--color-background-primary);color:var(--color-text-secondary);transition:all 0.15s;}
.lp-kbtn.active{background:#FBEAF0;border-color:#993556;color:#72243E;font-weight:500;}
</style>

<h2 class="sr-only">Simulador do operador Laplaciano para realce de bordas: imagem original, laplaciano e resultado.</h2>

<div style="max-width:700px;margin:0 auto;padding:8px 0 16px;">

  <div style="background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-lg);padding:10px 14px;margin-bottom:12px;display:flex;align-items:center;gap:10px;flex-wrap:wrap;">
    <span style="font-size:12px;font-weight:500;color:var(--color-text-secondary);">Variante:</span>
    <button class="lp-kbtn active" id="lp-btn-v1">g = f − ∇²f <span style="font-weight:400;font-size:10px;">(realce padrão)</span></button>
    <button class="lp-kbtn" id="lp-btn-v2">g = f + ∇²f <span style="font-weight:400;font-size:10px;">(inverte sinal)</span></button>
    <button id="lp-btn-new" style="margin-left:auto;font-size:12px;padding:6px 12px;">🎲 Novo degrau</button>
  </div>

  <div style="display:grid;grid-template-columns:1fr 28px 1fr;gap:6px;align-items:center;margin-bottom:8px;">
    <div class="lp-card">
      <p class="lp-label" style="color:var(--color-text-secondary);">① Imagem original f</p>
      <p class="lp-sublabel" style="color:var(--color-text-secondary);">degrau com ruído leve</p>
      <div id="lp-gf" class="lp-grid"></div>
    </div>
    <div style="display:flex;align-items:center;justify-content:center;font-size:18px;color:var(--color-text-secondary);">→</div>
    <div class="lp-card" style="border:1.5px solid #185FA5;">
      <p class="lp-label" style="color:#185FA5;">③ Resultado g</p>
      <p class="lp-sublabel" style="color:#185FA5;" id="lp-flabel">g = f − ∇²f</p>
      <div id="lp-gr" class="lp-grid"></div>
    </div>
  </div>

  <div style="display:grid;grid-template-columns:1fr 28px 1fr;gap:6px;align-items:center;margin-bottom:10px;">
    <div class="lp-card" style="border:1.5px solid #993556;">
      <p class="lp-label" style="color:#993556;">② Laplaciano ∇²f</p>
      <p class="lp-sublabel" style="color:#993556;">bordas detectadas (±128 shift)</p>
      <div id="lp-gl" class="lp-grid"></div>
    </div>
    <div style="display:flex;align-items:center;justify-content:center;font-size:18px;color:var(--color-text-secondary);padding-top:0;"></div>
    <div style="background:var(--color-background-secondary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-lg);padding:10px 12px;font-size:11px;color:var(--color-text-secondary);line-height:1.8;font-family:var(--font-mono);">
      <div style="margin-bottom:4px;font-weight:500;font-size:10px;text-transform:uppercase;letter-spacing:.04em;">Kernel w4</div>
      <div style="display:inline-grid;grid-template-columns:repeat(3,26px);gap:2px;margin-bottom:6px;">
        <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;background:var(--color-background-primary);border-radius:3px;color:var(--color-text-secondary);">0</div>
        <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;background:#FBEAF0;border-radius:3px;color:#72243E;">+1</div>
        <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;background:var(--color-background-primary);border-radius:3px;color:var(--color-text-secondary);">0</div>
        <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;background:#FBEAF0;border-radius:3px;color:#72243E;">+1</div>
        <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;background:#993556;border-radius:3px;color:#fff;">−4</div>
        <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;background:#FBEAF0;border-radius:3px;color:#72243E;">+1</div>
        <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;background:var(--color-background-primary);border-radius:3px;color:var(--color-text-secondary);">0</div>
        <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;background:#FBEAF0;border-radius:3px;color:#72243E;">+1</div>
        <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;background:var(--color-background-primary);border-radius:3px;color:var(--color-text-secondary);">0</div>
      </div>
      <div style="font-size:10px;color:var(--color-text-secondary);">∇²f = T+B+L+R − 4·f</div>
    </div>
  </div>

  <div style="background:var(--color-background-secondary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-lg);padding:8px 14px;display:flex;gap:14px;align-items:center;flex-wrap:wrap;justify-content:center;margin-bottom:8px;">
    <span style="font-size:11px;font-weight:500;color:var(--color-text-secondary);">Legenda</span>
    <div style="display:flex;align-items:center;gap:5px;"><div style="width:14px;height:14px;background:#FAEEDA;border:1.5px dashed #BA7517;border-radius:3px;"></div><span style="font-size:11px;color:var(--color-text-secondary);">4-vizinhos do kernel</span></div>
    <div style="display:flex;align-items:center;gap:5px;"><div style="width:14px;height:14px;background:#FAECE7;border:1.5px solid #993C1D;border-radius:3px;"></div><span style="font-size:11px;color:var(--color-text-secondary);">Pixel central inspecionado</span></div>
    <div style="display:flex;align-items:center;gap:5px;"><div style="width:14px;height:14px;background:#E1F5EE;border:1.5px solid #1D9E75;border-radius:3px;"></div><span style="font-size:11px;color:var(--color-text-secondary);">Borda — copiada sem filtro</span></div>
  </div>

  <div id="lp-debug" style="background:#E6F1FB;border:0.5px solid #B5D4F4;border-radius:var(--border-radius-md);padding:9px 14px;font-family:var(--font-mono);font-size:11px;color:#0C447C;text-align:center;min-height:36px;line-height:1.7;">
    Passe o mouse sobre um pixel interno do resultado para detalhar a equação.
  </div>
</div>

<script>
(function(){
  const R=5,C=5,N=25;
  let px=[],lapV=[],lapA=[],rs=[];
  let variant='subtract';

  function gen(){
    const sc=2+Math.floor(Math.random()*2);
    const dark=40+Math.floor(Math.random()*30);
    const light=160+Math.floor(Math.random()*40);
    px=Array.from({length:N},(_,i)=>{
      const c=i%C;
      let v=c<sc?dark:light;
      v+=Math.floor(Math.random()*14)-7;
      return Math.max(0,Math.min(255,v));
    });
    calc();
  }

  function calc(){
    lapV=new Array(N).fill(0);lapA=new Array(N).fill(0);rs=[...px];
    for(let r=0;r<R;r++){
      for(let c=0;c<C;c++){
        const i=r*C+c;
        if(r>=1&&r<R-1&&c>=1&&c<C-1){
          const t=px[(r-1)*C+c],b=px[(r+1)*C+c],l=px[r*C+(c-1)],ri=px[r*C+(c+1)],f=px[i];
          const lap=t+b+l+ri-4*f;
          lapV[i]=lap;
          lapA[i]=Math.max(0,Math.min(255,lap+128));
          rs[i]=Math.max(0,Math.min(255,variant==='subtract'?f-lap:f+lap));
        }
      }
    }
  }

  function tc(g){return g>140?'#111':'#eee';}

  function highlightCross(tr,tc2,on){
    const cells=document.getElementById('lp-gf').children;
    for(let i=0;i<N;i++){
      if(!cells[i])continue;
      const p=px[i];
      cells[i].style.background=`rgb(${p},${p},${p})`;cells[i].style.color=tc(p);cells[i].style.boxShadow='none';
    }
    if(on){
      const ci=tr*C+tc2;
      if(cells[ci]){cells[ci].style.background='#FAECE7';cells[ci].style.color='#712B13';cells[ci].style.boxShadow='0 0 0 2px #993C1D inset';}
      [[tr-1,tc2],[tr+1,tc2],[tr,tc2-1],[tr,tc2+1]].forEach(([nr,nc])=>{
        if(nr>=0&&nr<R&&nc>=0&&nc<C){const ni=nr*C+nc;if(cells[ni]){cells[ni].style.background='#FAEEDA';cells[ni].style.color='#633806';cells[ni].style.boxShadow='0 0 0 2px #BA7517 inset';}}
      });
    }
  }

  function resetDebug(){document.getElementById('lp-debug').innerText='Passe o mouse sobre um pixel interno do resultado para detalhar a equação.';}

  function render(){
    const gf=document.getElementById('lp-gf'),gl=document.getElementById('lp-gl'),gr=document.getElementById('lp-gr');
    gf.innerHTML='';gl.innerHTML='';gr.innerHTML='';
    document.getElementById('lp-flabel').innerText=variant==='subtract'?'g = f − ∇²f':'g = f + ∇²f';

    for(let i=0;i<N;i++){
      const r=Math.floor(i/C),c=i%C;
      const p=px[i],lv=lapV[i],la=lapA[i],res=rs[i];
      const border=(r===0||r===R-1||c===0||c===C-1);

      const cf=document.createElement('div');cf.className='lp-cell';
      cf.style.cssText=`background:rgb(${p},${p},${p});color:${tc(p)};border:0.5px solid rgba(128,128,128,0.15);`;
      cf.innerText=p;gf.appendChild(cf);

      const cl=document.createElement('div');cl.className='lp-cell';
      if(border){cl.style.cssText=`background:var(--color-background-secondary);color:var(--color-text-secondary);border:0.5px solid var(--color-border-tertiary);`;cl.innerText='—';}
      else{cl.style.cssText=`background:rgb(${la},${la},${la});color:${tc(la)};border:0.5px solid rgba(128,128,128,0.15);`;cl.innerText=lv;}
      gl.appendChild(cl);

      const crr=document.createElement('div');crr.className='lp-cell';
      if(border){
        crr.style.cssText=`background:#E1F5EE;color:#085041;border:1.5px solid #1D9E75;`;
        crr.onmouseenter=()=>{document.getElementById('lp-debug').innerHTML=`Pixel borda (${r},${c}): valor herdado sem cálculo → <b>${res}</b>`;};
        crr.onmouseleave=resetDebug;
      } else {
        crr.style.cssText=`background:rgb(${res},${res},${res});color:${tc(res)};border:0.5px solid var(--color-border-tertiary);cursor:pointer;`;
        crr.innerText=res;
        const _r=r,_c=c,_p=p,_lv=lv,_res=res;
        const _t=px[(_r-1)*C+_c],_b=px[(_r+1)*C+_c],_l=px[_r*C+(_c-1)],_ri=px[_r*C+(_c+1)];
        crr.onmouseenter=()=>{
          highlightCross(_r,_c,true);
          crr.style.transform='scale(1.12)';crr.style.boxShadow='0 0 0 2px #185FA5 inset';
          crr.style.background='#E6F1FB';crr.style.color='#0C447C';
          const sign=variant==='subtract'?'−':'+';
          document.getElementById('lp-debug').innerHTML=`∇²f = (${_t}+${_b}+${_l}+${_ri}) − 4·${_p} = <b>${_lv}</b> &nbsp;|&nbsp; g = clip(${_p} ${sign} ${_lv}) = <b>${_res}</b>`;
        };
        crr.onmouseleave=()=>{
          highlightCross(_r,_c,false);
          crr.style.transform='scale(1)';crr.style.boxShadow='none';
          crr.style.background=`rgb(${_res},${_res},${_res})`;crr.style.color=tc(_res);
          resetDebug();
        };
      }
      crr.innerText=res;
      gr.appendChild(crr);
    }
  }

  function setVariant(v){
    variant=v;
    document.getElementById('lp-btn-v1').classList.toggle('active',v==='subtract');
    document.getElementById('lp-btn-v2').classList.toggle('active',v==='add');
    calc();render();
  }

  document.getElementById('lp-btn-v1').onclick=()=>setVariant('subtract');
  document.getElementById('lp-btn-v2').onclick=()=>setVariant('add');
  document.getElementById('lp-btn-new').onclick=()=>{gen();render();};
  gen();render();
})();
</script>


""")

In [21]:
%%writefile EP03_07.py
# Código Python

Writing EP03_07.py


In [22]:
TestSuite("EP03_07.py").run()

---

### EP03_08 🧭 Gradiente de Sobel: Gx e Gy

Em robôs exploradores de Marte (como o Perseverance), a detecção de obstáculos é realizada em tempo real por câmeras estereoscópicas. O **operador de Sobel** calcula o gradiente direcional da cena e é utilizado no algoritmo de detecção de bordas para identificar rochas, fissuras e desníveis do terreno que possam comprometer a navegação.

Ver na @fig-03-sobel-simulador uma simulação deste EP.


#### 📋 Diretrizes de Implementação

1. **Dimensões:** Ler os inteiros $L$ (linhas) e $C$ (colunas).
2. **Dados:** Ler a matriz $f$.
3. **Gx e Gy:** Para cada pixel **interno** $(i,j)$ com $1 \le i < L-1$, $1 \le j < C-1$:

$$G_x(i,j) = [f(i-1,j+1) + 2f(i,j+1) + f(i+1,j+1)] - [f(i-1,j-1) + 2f(i,j-1) + f(i+1,j-1)]$$

$$G_y(i,j) = [f(i+1,j-1) + 2f(i+1,j) + f(i+1,j+1)] - [f(i-1,j-1) + 2f(i-1,j) + f(i-1,j+1)]$$

4. **Magnitude:** $|\nabla f(i,j)| = \text{clip}(\text{round}(\sqrt{G_x^2 + G_y^2}))$.
5. **Borda:** Pixels de borda recebem magnitude 0.
6. **Saída:** Exibir a magnitude $L \times C$.

---

#### 📌 Restrições Computacionais

* **Arredondamento:** Aplicar `round` antes de converter para inteiro.
* **Saturação:** $\text{clip}(x) = \max(0, \min(255, x))$.
* **Raiz quadrada:** Usar $\sqrt{G_x^2 + G_y^2}$ (não a aproximação $|G_x| + |G_y|$).

---

#### 🧠 Fundamentação Teórica

| Operador | Detecta | Coeficientes diagonais |
|:--------:|:-------:|:----------------------:|
| $G_x$ | Bordas verticais | $\pm 1$ |
| $G_y$ | Bordas horizontais | $\pm 1$ |
| $|\nabla f|$ | Todas as bordas | Combinado |

---

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linhas seguintes: Elementos da matriz.

**Saída:**

* Magnitude do gradiente, matriz $L \times C$.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 3<br>3<br>0 0 0<br>0 0 0<br>0 0 0 | 0 0 0<br>0 0 0<br>0 0 0 | Imagem nula: gradiente zero |
| 3<br>3<br>0 0 255<br>0 0 255<br>0 0 255 | 0 0 0<br>0 255 0<br>0 0 0 | Borda vertical central: Gx alto |

In [6]:
#| label: fig-03-sobel-simulador
#| fig-cap: "Simulador: Gradiente de Sobel: Gx e Gy"
#| echo: false
#| output: true

from IPython.display import HTML
HTML("""

<style>
.sb-cell{width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9px;font-weight:500;border-radius:4px;transition:background 0.1s,transform 0.12s;}
.sb-card{background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-lg);padding:12px 14px;text-align:center;}
.sb-label{font-size:11px;font-weight:500;text-transform:uppercase;letter-spacing:.05em;margin:0 0 2px;color:var(--color-text-secondary);}
.sb-sublabel{font-size:10px;margin:0 0 8px;color:var(--color-text-secondary);}
.sb-grid{display:grid;grid-template-columns:repeat(5,36px);gap:3px;justify-content:center;margin-top:6px;}
.sb-kgrid{display:inline-grid;grid-template-columns:repeat(3,26px);gap:2px;margin:0 2px;}
.sb-kc{width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:500;border-radius:3px;font-family:var(--font-mono);}
</style>

<h2 class="sr-only">Simulador do gradiente de Sobel: imagem original, Gx, Gy e magnitude do gradiente.</h2>

<div style="max-width:700px;margin:0 auto;padding:8px 0 16px;">

  <div style="background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-lg);padding:10px 14px;margin-bottom:12px;display:flex;align-items:center;gap:12px;flex-wrap:wrap;">
    <button id="sb-btn" style="font-size:12px;padding:6px 12px;">🎲 Nova cena</button>
    <span style="font-size:12px;color:var(--color-text-secondary);margin-left:4px;">Kernels de Sobel:</span>
    <div style="display:flex;align-items:center;gap:6px;">
      <div class="sb-kgrid" style="background:var(--color-background-secondary);border-radius:5px;padding:4px;">
        <div class="sb-kc" style="background:#0C447C;color:#B5D4F4;">−1</div><div class="sb-kc" style="background:var(--color-background-primary);color:var(--color-text-secondary);">0</div><div class="sb-kc" style="background:#185FA5;color:#E6F1FB;">+1</div>
        <div class="sb-kc" style="background:#0C447C;color:#B5D4F4;">−2</div><div class="sb-kc" style="background:var(--color-background-primary);color:var(--color-text-secondary);">0</div><div class="sb-kc" style="background:#185FA5;color:#E6F1FB;">+2</div>
        <div class="sb-kc" style="background:#0C447C;color:#B5D4F4;">−1</div><div class="sb-kc" style="background:var(--color-background-primary);color:var(--color-text-secondary);">0</div><div class="sb-kc" style="background:#185FA5;color:#E6F1FB;">+1</div>
      </div>
      <span style="font-size:11px;font-weight:500;color:#185FA5;font-family:var(--font-mono);">Gx</span>
    </div>
    <div style="display:flex;align-items:center;gap:6px;">
      <div class="sb-kgrid" style="background:var(--color-background-secondary);border-radius:5px;padding:4px;">
        <div class="sb-kc" style="background:#633806;color:#FAC775;">−1</div><div class="sb-kc" style="background:#854F0B;color:#FAC775;">−2</div><div class="sb-kc" style="background:#633806;color:#FAC775;">−1</div>
        <div class="sb-kc" style="background:var(--color-background-primary);color:var(--color-text-secondary);">0</div><div class="sb-kc" style="background:var(--color-background-primary);color:var(--color-text-secondary);">0</div><div class="sb-kc" style="background:var(--color-background-primary);color:var(--color-text-secondary);">0</div>
        <div class="sb-kc" style="background:#BA7517;color:#FAEEDA;">+1</div><div class="sb-kc" style="background:#EF9F27;color:#412402;">+2</div><div class="sb-kc" style="background:#BA7517;color:#FAEEDA;">+1</div>
      </div>
      <span style="font-size:11px;font-weight:500;color:#BA7517;font-family:var(--font-mono);">Gy</span>
    </div>
  </div>

  <div style="display:grid;grid-template-columns:1fr 28px 1fr;gap:6px;align-items:center;margin-bottom:8px;">
    <div class="sb-card">
      <p class="sb-label">Imagem original f</p>
      <p class="sb-sublabel">5 × 5 pixels</p>
      <div id="sb-gf" class="sb-grid"></div>
    </div>
    <div style="display:flex;align-items:center;justify-content:center;font-size:18px;color:var(--color-text-secondary);">→</div>
    <div class="sb-card" style="border:1.5px solid #0F6E56;">
      <p class="sb-label" style="color:#0F6E56;">|∇f| magnitude</p>
      <p class="sb-sublabel" style="color:#0F6E56;">√(Gx² + Gy²)</p>
      <div id="sb-gm" class="sb-grid"></div>
    </div>
  </div>

  <div style="display:grid;grid-template-columns:1fr 28px 1fr;gap:6px;align-items:center;margin-bottom:10px;">
    <div class="sb-card" style="border:1.5px solid #185FA5;">
      <p class="sb-label" style="color:#185FA5;">Gx — gradiente horizontal</p>
      <p class="sb-sublabel" style="color:#185FA5;">azul escuro = neg · branco = zero · azul vivo = pos</p>
      <div id="sb-ggx" class="sb-grid"></div>
    </div>
    <div style="display:flex;align-items:center;justify-content:center;font-size:18px;color:var(--color-text-secondary);">⊕</div>
    <div class="sb-card" style="border:1.5px solid #BA7517;">
      <p class="sb-label" style="color:#BA7517;">Gy — gradiente vertical</p>
      <p class="sb-sublabel" style="color:#BA7517;">âmbar escuro = neg · branco = zero · âmbar vivo = pos</p>
      <div id="sb-ggy" class="sb-grid"></div>
    </div>
  </div>

  <div style="background:var(--color-background-secondary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-lg);padding:8px 14px;display:flex;gap:14px;align-items:center;flex-wrap:wrap;justify-content:center;margin-bottom:8px;">
    <span style="font-size:11px;font-weight:500;color:var(--color-text-secondary);">Legenda</span>
    <div style="display:flex;align-items:center;gap:5px;"><div style="width:14px;height:14px;background:#FAEEDA;border:1.5px dashed #BA7517;border-radius:3px;"></div><span style="font-size:11px;color:var(--color-text-secondary);">Vizinhança 3×3 inspecionada</span></div>
    <div style="display:flex;align-items:center;gap:5px;"><div style="width:14px;height:14px;background:#E6F1FB;border:1.5px solid #185FA5;border-radius:3px;"></div><span style="font-size:11px;color:var(--color-text-secondary);">Pixel central</span></div>
    <div style="display:flex;align-items:center;gap:5px;"><div style="width:14px;height:14px;background:#2C2C2A;border:1.5px solid #993556;border-radius:3px;"></div><span style="font-size:11px;color:var(--color-text-secondary);">Borda — forçada para 0</span></div>
  </div>

  <div id="sb-debug" style="background:#E6F1FB;border:0.5px solid #B5D4F4;border-radius:var(--border-radius-md);padding:9px 14px;font-family:var(--font-mono);font-size:11px;color:#0C447C;text-align:center;min-height:36px;line-height:1.7;">
    Passe o mouse sobre um pixel interno da magnitude para ver a decomposição Gx e Gy.
  </div>
</div>

<script>
(function(){
  const R=5,C=5,N=25;
  let px=[],gxV=[],gyV=[],mgV=[];

  function gen(){
    const block=Math.random()>0.3;
    const bg=30+Math.floor(Math.random()*30);
    const obj=180+Math.floor(Math.random()*50);
    px=Array.from({length:N},(_,i)=>{
      const r=Math.floor(i/C),c=i%C;
      let v=bg;
      if(block&&r>=2&&r<=3&&c>=2&&c<=3)v=obj;
      else if(!block&&r+c>=4)v=obj-40;
      v+=Math.floor(Math.random()*10)-5;
      return Math.max(0,Math.min(255,v));
    });
    calc();
  }

  function calc(){
    gxV=new Array(N).fill(0);gyV=new Array(N).fill(0);mgV=new Array(N).fill(0);
    for(let r=0;r<R;r++){
      for(let c=0;c<C;c++){
        const i=r*C+c;
        if(r>=1&&r<R-1&&c>=1&&c<C-1){
          const p=[[r-1,c-1],[r-1,c],[r-1,c+1],[r,c-1],[r,c],[r,c+1],[r+1,c-1],[r+1,c],[r+1,c+1]].map(([rr,cc])=>px[rr*C+cc]);
          const gx=(p[2]+2*p[5]+p[8])-(p[0]+2*p[3]+p[6]);
          const gy=(p[6]+2*p[7]+p[8])-(p[0]+2*p[1]+p[2]);
          gxV[i]=gx;gyV[i]=gy;mgV[i]=Math.min(255,Math.round(Math.sqrt(gx*gx+gy*gy)));
        }
      }
    }
  }

  function tc(g){return g>150?'#111':'#eee';}

  function colorGx(v){
    const n=Math.max(-510,Math.min(510,v));
    if(n>=0){
      const t=Math.min(1,n/300);
      return `rgb(${Math.round(255-t*210)},${Math.round(255-t*170)},${Math.round(255-t*50)})`;
    } else {
      const t=Math.min(1,-n/300);
      return `rgb(${Math.round(255-t*210)},${Math.round(255-t*150)},${Math.round(255-t*50)})`;
    }
  }

  function colorGxBetter(v){
    const n=Math.max(-400,Math.min(400,v));
    if(Math.abs(n)<15) return '#f5f5f3';
    if(n>0){
      const t=Math.min(1,n/350);
      const r=Math.round(4+t*20);
      const g=Math.round(44+t*71);
      const b=Math.round(83+t*89);
      return `rgb(${r},${g},${b})`;
    } else {
      const t=Math.min(1,-n/350);
      return `rgb(${Math.round(12+t*0)},${Math.round(68-t*24)},${Math.round(165-t*82)})`;
    }
  }

  function colorGyBetter(v){
    const n=Math.max(-400,Math.min(400,v));
    if(Math.abs(n)<15) return '#f5f5f3';
    if(n>0){
      const t=Math.min(1,n/350);
      return `rgb(${Math.round(186+t*63)},${Math.round(117+t*70)},${Math.round(23-t*18)})`;
    } else {
      const t=Math.min(1,-n/350);
      return `rgb(${Math.round(99+t*0)},${Math.round(56-t*20)},${Math.round(11-t*5)})`;
    }
  }

  function tcSigned(bg){
    const m=bg.match(/rgb\((\d+),(\d+),(\d+)\)/);
    if(!m)return'#111';
    const lum=0.299*m[1]+0.587*m[2]+0.114*m[3];
    return lum>140?'#111':'#eee';
  }

  function highlight(tr,tc2,on){
    const cells=document.getElementById('sb-gf').children;
    for(let i=0;i<N;i++){
      if(!cells[i])continue;
      const p=px[i],r=Math.floor(i/C),c=i%C;
      if(on&&Math.abs(r-tr)<=1&&Math.abs(c-tc2)<=1){
        if(r===tr&&c===tc2){cells[i].style.background='#E6F1FB';cells[i].style.color='#0C447C';cells[i].style.boxShadow='0 0 0 2px #185FA5 inset';}
        else{cells[i].style.background='#FAEEDA';cells[i].style.color='#633806';cells[i].style.boxShadow='0 0 0 2px #BA7517 inset';}
      } else {cells[i].style.background=`rgb(${p},${p},${p})`;cells[i].style.color=tc(p);cells[i].style.boxShadow='none';}
    }
  }

  function resetDebug(){document.getElementById('sb-debug').innerText='Passe o mouse sobre um pixel interno da magnitude para ver a decomposição Gx e Gy.';}

  function render(){
    const gf=document.getElementById('sb-gf'),ggx=document.getElementById('sb-ggx'),ggy=document.getElementById('sb-ggy'),gm=document.getElementById('sb-gm');
    gf.innerHTML='';ggx.innerHTML='';ggy.innerHTML='';gm.innerHTML='';

    for(let i=0;i<N;i++){
      const r=Math.floor(i/C),c=i%C;
      const p=px[i],gx=gxV[i],gy=gyV[i],mag=mgV[i];
      const border=(r===0||r===R-1||c===0||c===C-1);

      const cf=document.createElement('div');cf.className='sb-cell';
      cf.style.cssText=`background:rgb(${p},${p},${p});color:${tc(p)};border:0.5px solid rgba(128,128,128,0.15);`;
      cf.innerText=p;gf.appendChild(cf);

      const cgx=document.createElement('div');cgx.className='sb-cell';
      if(border){cgx.style.cssText=`background:var(--color-background-secondary);color:var(--color-text-secondary);border:0.5px solid var(--color-border-tertiary);`;cgx.innerText='—';}
      else{const bg=colorGxBetter(gx);cgx.style.cssText=`background:${bg};color:${tcSigned(bg)};border:0.5px solid rgba(128,128,128,0.1);`;cgx.innerText=(gx>0?'+':'')+gx;}
      ggx.appendChild(cgx);

      const cgy=document.createElement('div');cgy.className='sb-cell';
      if(border){cgy.style.cssText=`background:var(--color-background-secondary);color:var(--color-text-secondary);border:0.5px solid var(--color-border-tertiary);`;cgy.innerText='—';}
      else{const bg=colorGyBetter(gy);cgy.style.cssText=`background:${bg};color:${tcSigned(bg)};border:0.5px solid rgba(128,128,128,0.1);`;cgy.innerText=(gy>0?'+':'')+gy;}
      ggy.appendChild(cgy);

      const cm=document.createElement('div');cm.className='sb-cell';
      if(border){
        cm.style.cssText=`background:#2C2C2A;color:#9A9892;border:1.5px solid #993556;`;cm.innerText='0';
        cm.onmouseenter=()=>{document.getElementById('sb-debug').innerHTML=`Pixel borda (${r},${c}): sem vizinhança completa → forçado para <b>0</b>`;};
        cm.onmouseleave=resetDebug;
      } else {
        cm.style.cssText=`background:rgb(${mag},${mag},${mag});color:${tc(mag)};border:0.5px solid var(--color-border-tertiary);cursor:pointer;`;
        cm.innerText=mag;
        const _r=r,_c=c,_gx=gx,_gy=gy,_mag=mag;
        cm.onmouseenter=()=>{
          highlight(_r,_c,true);
          cm.style.transform='scale(1.12)';cm.style.boxShadow='0 0 0 2px #0F6E56 inset';
          cm.style.background='#E1F5EE';cm.style.color='#085041';
          const gxs=_gx>=0?'+':'';const gys=_gy>=0?'+':'';
          document.getElementById('sb-debug').innerHTML=`Pixel (${_r},${_c}): Gx = <b>${_gx}</b> &nbsp;|&nbsp; Gy = <b>${_gy}</b> &nbsp;|&nbsp; |∇f| = round(√(${_gx}²+${_gy}²)) = <b>${_mag}</b>`;
        };
        cm.onmouseleave=()=>{
          highlight(_r,_c,false);
          cm.style.transform='scale(1)';cm.style.boxShadow='none';
          cm.style.background=`rgb(${_mag},${_mag},${_mag})`;cm.style.color=tc(_mag);
          resetDebug();
        };
      }
      gm.appendChild(cm);
    }
  }

  document.getElementById('sb-btn').onclick=()=>{gen();render();};
  gen();render();
})();
</script>



""")

<>:160: SyntaxWarning: invalid escape sequence '\('
<>:160: SyntaxWarning: invalid escape sequence '\('
/tmp/ipykernel_17250/3406689444.py:160: SyntaxWarning: invalid escape sequence '\('
  const m=bg.match(/rgb\((\d+),(\d+),(\d+)\)/);


In [24]:
%%writefile EP03_08.py
# Código Python

Writing EP03_08.py


In [25]:
TestSuite("EP03_08.py").run()

---

### EP03_09 📡 Filtro da Mediana 3×3

Imagens de radar de abertura sintética (SAR) usadas em monitoramento ambiental e militar sofrem de um tipo específico de ruído chamado *speckle*, que possui características similares ao ruído sal e pimenta. O **filtro da mediana** é o método padrão de remoção desse ruído porque preserva as bordas das estruturas enquanto elimina os pontos espúrios.

Ver na @fig-03-median-simulador uma simulação deste EP.


#### 📋 Diretrizes de Implementação

1. **Dimensões:** Ler os inteiros $L$ (linhas) e $C$ (colunas).
2. **Dados:** Ler a matriz de pixels $f$.
3. **Filtro da Mediana 3×3:** Para cada pixel **interno** $(i,j)$ com $1 \le i < L-1$, $1 \le j < C-1$:
   - Coletar os 9 pixels da vizinhança $3 \times 3$: $\{f(i+s, j+t) : s,t \in \{-1,0,1\}\}$.
   - Ordenar os 9 valores em ordem crescente.
   - Atribuir $g(i,j)$ ao valor central (posição índice 4, considerando índice 0).

$$g(i,j) = \text{mediana}\{f(i+s, j+t) : s,t \in \{-1,0,1\}\}$$

4. **Borda:** Copiar diretamente: $g(i,j) = f(i,j)$.
5. **Saída:** Exibir a matriz filtrada $L \times C$.

---

#### 📌 Restrições Computacionais

* **Janela:** Sempre $3 \times 3 = 9$ elementos.
* **Mediana:** O elemento central da sequência ordenada (índice 4 de 0 a 8).
* **Sem clipping:** A mediana de valores em $[0, 255]$ permanece em $[0, 255]$.
* **Não linear:** O filtro mediana não pode ser expresso como convolução linear.

---

#### 🧠 Fundamentação Teórica

| Ruído | Filtro de Média | Filtro de Mediana |
|:-----:|:---------------:|:-----------------:|
| Sal e pimenta (0 ou 255) | Espalha o ruído | Remove sem distorcer bordas |
| Gaussiano | Reduz eficazmente | Reduz parcialmente |

---

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linhas seguintes: Elementos da matriz.

**Saída:**

* Matriz filtrada $L \times C$.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 3<br>3<br>100 100 100<br>100 0 100<br>100 100 100 | 100 100 100<br>100 100 100<br>100 100 100 | Ponto preto eliminado: mediana de 8×100+1×0 = 100 |
| 3<br>3<br>50 50 50<br>50 255 50<br>50 50 50 | 50 50 50<br>50 50 50<br>50 50 50 | Ponto branco (sal) eliminado |

In [5]:
#| label: fig-03-median-simulador
#| fig-cap: "Simulador: Filtro da Mediana 3×3"
#| echo: false
#| output: true

from IPython.display import HTML
HTML("""
<style>
.ep0309-cell{width:38px;height:38px;display:flex;align-items:center;justify-content:center;font-size:9px;font-weight:500;border-radius:5px;transition:background 0.1s,transform 0.12s,box-shadow 0.1s;}
.ep0309-card{background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-lg);padding:14px;text-align:center;}
.ep0309-label{font-size:10px;font-weight:500;color:var(--color-text-secondary);text-transform:uppercase;letter-spacing:.05em;margin:0 0 3px;}
.ep0309-sublabel{font-size:10px;margin:0 0 8px;}
.ep0309-grid{display:grid;grid-template-columns:repeat(5,38px);gap:3px;justify-content:center;margin-top:8px;}
.ep0309-vitem{display:inline-flex;align-items:center;justify-content:center;width:32px;height:24px;border-radius:4px;font-size:10px;font-weight:500;font-family:var(--font-mono);margin:2px;}
</style>

<div id="ep0309-sim-mediana" style="max-width:960px;margin:0 auto;padding:4px 0 16px;">

  <div style="display:flex;align-items:center;gap:10px;margin-bottom:14px;flex-wrap:wrap;">
    <button id="ep0309-btn-new" style="font-size:12px;padding:6px 14px;">
      🎲 Injetar ruído impulsivo
    </button>
    <span style="font-size:11px;color:var(--color-text-secondary);">Ruído sal (255) e pimenta (0) — ~30% dos pixels internos afetados</span>
  </div>

  <!-- Grades f e g -->
  <div style="display:grid;grid-template-columns:1fr 1fr;gap:14px;margin-bottom:12px;">

    <div class="ep0309-card">
      <p class="ep0309-label">Imagem f — com ruído</p>
      <p class="ep0309-sublabel" style="color:var(--color-text-secondary);">sal (255) e pimenta (0) visíveis</p>
      <div id="ep0309-grid-f" class="ep0309-grid"></div>
    </div>

    <div class="ep0309-card" style="border:2px solid #185FA5;">
      <p class="ep0309-label" style="color:#185FA5;">Resultado g — sem ruído</p>
      <p class="ep0309-sublabel" style="color:#185FA5;">passe o mouse para inspecionar</p>
      <div id="ep0309-grid-result" class="ep0309-grid"></div>
    </div>
  </div>

  <!-- Vetor ordenado -->
  <div class="ep0309-card" style="margin-bottom:12px;">
    <p class="ep0309-label">Vetor de vizinhança 3×3 — ordenado</p>
    <p class="ep0309-sublabel" style="color:var(--color-text-secondary);">passe o mouse num pixel interno do resultado para ver</p>
    <div id="ep0309-vector" style="display:flex;flex-wrap:wrap;justify-content:center;gap:2px;min-height:32px;padding:4px 0;align-items:center;">
      <span style="font-size:11px;color:var(--color-text-secondary);font-style:italic;">—</span>
    </div>
  </div>

  <!-- Legenda -->
  <div style="background:var(--color-background-secondary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-lg);padding:10px 14px;display:flex;gap:18px;align-items:center;flex-wrap:wrap;justify-content:center;margin-bottom:10px;">
    <span style="font-size:11px;font-weight:500;color:var(--color-text-secondary);">Legenda</span>
    <div style="display:flex;align-items:center;gap:6px;">
      <div style="width:18px;height:18px;background:#FAEEDA;border:2px dashed #BA7517;border-radius:3px;"></div>
      <span style="font-size:11px;color:var(--color-text-secondary);">Janela 3×3 capturada</span>
    </div>
    <div style="display:flex;align-items:center;gap:6px;">
      <div style="width:18px;height:18px;background:#E6F1FB;border:1.5px solid #185FA5;border-radius:3px;"></div>
      <span style="font-size:11px;color:var(--color-text-secondary);">Pixel central inspecionado</span>
    </div>
    <div style="display:flex;align-items:center;gap:6px;">
      <div style="width:18px;height:18px;background:#E1F5EE;border:1.5px solid #1D9E75;border-radius:3px;"></div>
      <span style="font-size:11px;color:var(--color-text-secondary);">Borda — copiada sem filtro</span>
    </div>
    <div style="display:flex;align-items:center;gap:6px;">
      <div style="width:18px;height:18px;background:#E0F5EF;border:2px solid #0F6E56;border-radius:3px;"></div>
      <span style="font-size:11px;color:var(--color-text-secondary);">Mediana (posição central)</span>
    </div>
    <div style="display:flex;align-items:center;gap:6px;">
      <div style="width:18px;height:18px;background:#2C2C2A;border:1.5px solid #B54030;border-radius:3px;"></div>
      <span style="font-size:11px;color:var(--color-text-secondary);">Ruído (0 ou 255) — eliminado</span>
    </div>
  </div>

  <!-- Debug -->
  <div id="ep0309-debug" style="background:#E6F1FB;border:0.5px solid #B5D4F4;border-radius:var(--border-radius-md);padding:10px 14px;font-family:var(--font-mono);font-size:11px;color:#0C447C;text-align:center;min-height:38px;line-height:1.7;">
    Passe o mouse sobre um pixel interno do resultado para ver o processo de ordenação.
  </div>
</div>

<script>
(function(){
  const EP0309_ROWS=5,EP0309_COLS=5,EP0309_N=25,EP0309_RADIUS=1;
  let ep0309Pixels=[],ep0309Result=[];

  function ep0309Gen(){
    ep0309Pixels=Array.from({length:EP0309_N},()=>{
      let v=110+Math.floor(Math.random()*30);
      const rnd=Math.random();
      if(rnd>0.82) v=255;
      else if(rnd<0.18) v=0;
      return v;
    });
    ep0309Calc();
  }

  function ep0309Calc(){
    ep0309Result=[...ep0309Pixels];
    for(let r=0;r<EP0309_ROWS;r++){
      for(let c=0;c<EP0309_COLS;c++){
        if(r>=EP0309_RADIUS&&r<EP0309_ROWS-EP0309_RADIUS&&c>=EP0309_RADIUS&&c<EP0309_COLS-EP0309_RADIUS){
          const win=[];
          for(let s=-EP0309_RADIUS;s<=EP0309_RADIUS;s++)
            for(let t=-EP0309_RADIUS;t<=EP0309_RADIUS;t++)
              win.push(ep0309Pixels[(r+s)*EP0309_COLS+(c+t)]);
          win.sort((a,b)=>a-b);
          ep0309Result[r*EP0309_COLS+c]=win[4];
        }
      }
    }
  }

  function ep0309TC(g){return g>140?'#111':'#eee';}

  function ep0309IsNoise(v){return v===0||v===255;}

  function ep0309Highlight(tr,tc,on){
    const cells=document.getElementById('ep0309-grid-f').children;
    for(let i=0;i<EP0309_N;i++){
      if(!cells[i]) continue;
      const p=ep0309Pixels[i];
      const r=Math.floor(i/EP0309_COLS),c=i%EP0309_COLS;
      if(on&&Math.abs(r-tr)<=1&&Math.abs(c-tc)<=1){
        if(r===tr&&c===tc){
          cells[i].style.background='#E6F1FB';cells[i].style.color='#0C447C';cells[i].style.boxShadow='0 0 0 2px #185FA5 inset';
        } else {
          cells[i].style.background='#FAEEDA';cells[i].style.color='#633806';cells[i].style.boxShadow='0 0 0 2px #BA7517 inset';
        }
      } else {
        cells[i].style.background=`rgb(${p},${p},${p})`;cells[i].style.color=ep0309TC(p);cells[i].style.boxShadow='none';
      }
    }
  }

  function ep0309ShowVector(raw,sorted,median){
    const el=document.getElementById('ep0309-vector');
    el.innerHTML='';
    // raw label
    const rawLabel=document.createElement('span');
    rawLabel.style.cssText='font-size:10px;color:var(--color-text-secondary);margin-right:4px;';
    rawLabel.innerText='Bruto:';
    el.appendChild(rawLabel);
    raw.forEach(v=>{
      const d=document.createElement('div');d.className='ep0309-vitem';
      const isN=ep0309IsNoise(v);
      d.style.background=isN?'#2C2C2A':'var(--color-background-secondary)';
      d.style.color=isN?'#E57373':'var(--color-text-primary)';
      d.style.border=isN?'1.5px solid #B54030':'0.5px solid var(--color-border-tertiary)';
      d.innerText=v;el.appendChild(d);
    });
    // arrow
    const arr=document.createElement('span');
    arr.style.cssText='font-size:16px;margin:0 6px;color:var(--color-text-secondary);';
    arr.innerText='→';el.appendChild(arr);
    // sorted label
    const sortLabel=document.createElement('span');
    sortLabel.style.cssText='font-size:10px;color:var(--color-text-secondary);margin-right:4px;';
    sortLabel.innerText='Ordenado:';
    el.appendChild(sortLabel);
    sorted.forEach((v,i)=>{
      const d=document.createElement('div');d.className='ep0309-vitem';
      const isMedian=(i===4);
      const isN=ep0309IsNoise(v);
      if(isMedian){
        d.style.background='#E0F5EF';d.style.color='#0A4A36';
        d.style.border='2px solid #0F6E56';d.style.fontWeight='700';
      } else if(isN){
        d.style.background='#2C2C2A';d.style.color='#E57373';
        d.style.border='1.5px solid #B54030';
      } else {
        d.style.background='var(--color-background-secondary)';
        d.style.color='var(--color-text-primary)';
        d.style.border='0.5px solid var(--color-border-tertiary)';
      }
      d.innerText=v;el.appendChild(d);
    });
    // result
    const eq=document.createElement('span');
    eq.style.cssText='font-size:11px;margin-left:6px;font-family:var(--font-mono);color:#0F6E56;font-weight:500;';
    eq.innerHTML=`→ mediana = <b>${median}</b>`;
    el.appendChild(eq);
  }

  function ep0309ResetVector(){
    document.getElementById('ep0309-vector').innerHTML=
      '<span style="font-size:11px;color:var(--color-text-secondary);font-style:italic;">—</span>';
  }

  function ep0309ResetDebug(){
    document.getElementById('ep0309-debug').innerText=
      'Passe o mouse sobre um pixel interno do resultado para ver o processo de ordenação.';
  }

  function ep0309Render(){
    const gf=document.getElementById('ep0309-grid-f');
    const gr=document.getElementById('ep0309-grid-result');
    gf.innerHTML='';gr.innerHTML='';

    for(let i=0;i<EP0309_N;i++){
      const r=Math.floor(i/EP0309_COLS),c=i%EP0309_COLS;
      const p=ep0309Pixels[i],res=ep0309Result[i];
      const isBorder=(r<EP0309_RADIUS||r>=EP0309_ROWS-EP0309_RADIUS||c<EP0309_RADIUS||c>=EP0309_COLS-EP0309_RADIUS);
      const isNoise=ep0309IsNoise(p);

      // f
      const cf=document.createElement('div');cf.className='ep0309-cell';
      if(isNoise){
        cf.style.cssText=`background:${p===255?'#fff':'#111'};color:${p===255?'#C0392B':'#E57373'};border:2px solid ${p===255?'#E74C3C':'#922B21'};font-weight:700;`;
      } else {
        cf.style.cssText=`background:rgb(${p},${p},${p});color:${ep0309TC(p)};border:0.5px solid rgba(128,128,128,0.15);`;
      }
      cf.innerText=p;gf.appendChild(cf);

      // result
      const crr=document.createElement('div');crr.className='ep0309-cell';
      if(isBorder){
        crr.style.cssText=`background:#E1F5EE;color:#085041;border:1.5px solid #1D9E75;`;
        crr.onmouseenter=()=>{
          document.getElementById('ep0309-debug').innerHTML=`Pixel borda (${r},${c}): copiado sem filtro → <b>${res}</b>`;
          ep0309ResetVector();
        };
        crr.onmouseleave=()=>{ep0309ResetDebug();};
      } else {
        crr.style.cssText=`background:rgb(${res},${res},${res});color:${ep0309TC(res)};border:0.5px solid var(--color-border-tertiary);cursor:pointer;`;
        const _r=r,_c=c,_p=p,_res=res;
        crr.onmouseenter=()=>{
          ep0309Highlight(_r,_c,true);
          crr.style.transform='scale(1.12)';crr.style.boxShadow='0 0 0 2px #185FA5 inset';
          crr.style.background='#E6F1FB';crr.style.color='#0C447C';
          const raw=[];
          for(let s=-EP0309_RADIUS;s<=EP0309_RADIUS;s++)
            for(let t=-EP0309_RADIUS;t<=EP0309_RADIUS;t++)
              raw.push(ep0309Pixels[(_r+s)*EP0309_COLS+(_c+t)]);
          const sorted=[...raw].sort((a,b)=>a-b);
          ep0309ShowVector(raw,sorted,_res);
          const noiseCount=raw.filter(ep0309IsNoise).length;
          document.getElementById('ep0309-debug').innerHTML=
            `Pixel (${_r},${_c}): ${noiseCount} vizinho(s) com ruído na janela &nbsp;→&nbsp; mediana = posição [4] = <b>${_res}</b>`;
        };
        crr.onmouseleave=()=>{
          ep0309Highlight(_r,_c,false);
          crr.style.transform='scale(1)';crr.style.boxShadow='none';
          crr.style.background=`rgb(${_res},${_res},${_res})`;crr.style.color=ep0309TC(_res);
          ep0309ResetDebug();ep0309ResetVector();
        };
      }
      crr.innerText=res;gr.appendChild(crr);
    }
  }

  document.getElementById('ep0309-btn-new').onclick=()=>{ep0309Gen();ep0309Render();};
  ep0309Gen();ep0309Render();
})();
</script>

""")

In [27]:
%%writefile EP03_09.py
# Código Python

Writing EP03_09.py


In [28]:
TestSuite("EP03_09.py").run()

---

### EP03_10 ✨ *Unsharp Masking* (USM)

Em sistemas de digitalização de documentos históricos e obras de arte, a nitidez das imagens é fundamental para leitura de textos manuscritos e detalhes ornamentais. O ***Unsharp Masking* (USM)** é o algoritmo de realce de nitidez padrão utilizado em *scanners* profissionais e softwares como Adobe Photoshop, controlado pelo parâmetro $k$ que determina a intensidade do realce.

Ver na @fig-03-unsharp-simulador uma simulação deste EP.


#### 📋 Diretrizes de Implementação

1. **Dimensões:** Ler os inteiros $L$ (linhas) e $C$ (colunas).
2. **Parâmetro:** Ler o valor real $k$ (intensidade do realce, $k \ge 0$).
3. **Dados:** Ler a matriz de pixels $f$.
4. **Suavização:** Calcular $\bar{f}$ com filtro de média $3\times3$ (apenas pixels internos; bordas mantidas):

$$\bar{f}(i,j) = \frac{1}{9} \sum_{s=-1}^{1} \sum_{t=-1}^{1} f(i+s, j+t)$$

5. **Máscara de alta frequência:** $m(i,j) = f(i,j) - \bar{f}(i,j)$.
6. **Realce USM:** Para cada pixel interno:

$$g(i,j) = \text{clip}\left(\text{round}\left(f(i,j) + k \cdot m(i,j)\right)\right)$$

7. **Borda:** $g(i,j) = f(i,j)$ (cópia direta).
8. **Saída:** Exibir a matriz realçada $L \times C$.

---

#### 📌 Restrições Computacionais

* **Arredondamento:** Aplicar `round` antes do clipping.
* **Saturação:** $\text{clip}(x) = \max(0, \min(255, x))$.
* **Operações em float:** Calcular $\bar{f}$ e $m$ em ponto flutuante antes de arredondar o resultado final.
* **$k = 0$:** Sem realce — a saída é idêntica à entrada (exceto pelas bordas).

---

#### 🧠 Fundamentação Teórica

| Etapa | Operação | Descrição |
|:-----:|:---------|:----------|
| 1 | $\bar{f} = f * \frac{1}{9}\mathbf{1}_{3\times3}$ | Suavização (baixas frequências) |
| 2 | $m = f - \bar{f}$ | Máscara (altas frequências) |
| 3 | $g = \text{clip}(\text{round}(f + k \cdot m))$ | Realce ponderado |

---

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linha 3: Real $k$.
* Linhas seguintes: Elementos da matriz original.

**Saída:**

* Matriz realçada $L \times C$.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 3<br>3<br>0.0<br>100 100 100<br>100 100 100<br>100 100 100 | 100 100 100<br>100 100 100<br>100 100 100 | k=0: sem realce |
| 3<br>3<br>1.0<br>50 50 50<br>50 200 50<br>50 50 50 | 50 50 50<br>50 255 50<br>50 50 50 | k=1: pixel central realçado e saturado |

In [3]:
#| label: fig-03-unsharp-simulador
#| fig-cap: "Simulador: *Unsharp Masking* (USM)"
#| echo: false
#| output: true

from IPython.display import HTML
HTML("""

<style>
.usm-cell{width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9px;font-weight:500;border-radius:4px;transition:background 0.1s,transform 0.12s;}
.usm-card{background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-lg);padding:12px 14px;text-align:center;}
.usm-label{font-size:11px;font-weight:500;color:var(--color-text-secondary);text-transform:uppercase;letter-spacing:.05em;margin:0 0 2px;}
.usm-sublabel{font-size:10px;margin:0 0 8px;color:var(--color-text-secondary);}
.usm-grid{display:grid;grid-template-columns:repeat(5,36px);gap:3px;justify-content:center;margin-top:6px;}
.usm-arrow{display:flex;align-items:center;justify-content:center;font-size:18px;color:var(--color-text-secondary);}
</style>

<h2 class="sr-only">Simulador de Unsharp Masking (USM): visualização do pipeline com imagem original, versão desfocada, máscara de alta frequência e resultado realçado.</h2>

<div style="max-width:700px;margin:0 auto;padding:8px 0 16px;">

  <div style="background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-lg);padding:12px 16px;margin-bottom:12px;display:flex;align-items:center;gap:12px;flex-wrap:wrap;">
    <span style="font-size:13px;font-weight:500;color:var(--color-text-primary);">Fator de ganho <em>k</em>:</span>
    <input id="usm-k" style="cursor:pointer;flex:1;min-width:140px;max-width:200px;accent-color:#185FA5;" max="3.0" min="0.0" step="0.5" type="range" value="1.0">
    <span id="usm-klabel" style="font-family:var(--font-mono);font-size:13px;font-weight:500;background:#185FA5;color:#fff;padding:2px 10px;border-radius:var(--border-radius-md);">k = 1.0</span>
    <span style="font-size:11px;color:var(--color-text-secondary);font-family:var(--font-mono);">0 = sem realce · 1 = USM padrão · &gt;1 = hiperrealce</span>
    <button id="usm-btn" style="margin-left:auto;font-size:12px;padding:6px 12px;">🎲 Nova imagem</button>
  </div>

  <div style="display:grid;grid-template-columns:1fr 28px 1fr;gap:6px;align-items:center;margin-bottom:8px;">
    <div class="usm-card">
      <p class="usm-label">① Imagem original f</p>
      <p class="usm-sublabel">5 × 5 pixels</p>
      <div id="usm-gf" class="usm-grid"></div>
    </div>
    <div class="usm-arrow">→</div>
    <div class="usm-card" style="border:1.5px solid #185FA5;">
      <p class="usm-label" style="color:#185FA5;">④ Resultado g</p>
      <p class="usm-sublabel" style="color:#185FA5;" id="usm-flabel">g = f + 1.0·m</p>
      <div id="usm-gr" class="usm-grid"></div>
    </div>
  </div>

  <div style="display:grid;grid-template-columns:1fr 28px 1fr;gap:6px;align-items:center;margin-bottom:10px;">
    <div class="usm-card" style="border:1.5px solid #BA7517;">
      <p class="usm-label" style="color:#BA7517;">② Desfocado f̄</p>
      <p class="usm-sublabel" style="color:#BA7517;">média 3 × 3</p>
      <div id="usm-gb" class="usm-grid"></div>
    </div>
    <div class="usm-arrow">→</div>
    <div class="usm-card" style="border:1.5px solid #993556;">
      <p class="usm-label" style="color:#993556;">③ Máscara m</p>
      <p class="usm-sublabel" style="color:#993556;">m = f − f̄ (altas freq.)</p>
      <div id="usm-gm" class="usm-grid"></div>
    </div>
  </div>

  <div style="background:var(--color-background-secondary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-lg);padding:8px 14px;display:flex;gap:14px;align-items:center;flex-wrap:wrap;justify-content:center;margin-bottom:8px;">
    <span style="font-size:11px;font-weight:500;color:var(--color-text-secondary);">Legenda</span>
    <div style="display:flex;align-items:center;gap:5px;"><div style="width:14px;height:14px;background:#FAEEDA;border:1.5px dashed #BA7517;border-radius:3px;"></div><span style="font-size:11px;color:var(--color-text-secondary);">Vizinhança 3×3</span></div>
    <div style="display:flex;align-items:center;gap:5px;"><div style="width:14px;height:14px;background:#E6F1FB;border:1.5px solid #185FA5;border-radius:3px;"></div><span style="font-size:11px;color:var(--color-text-secondary);">Pixel inspecionado</span></div>
    <div style="display:flex;align-items:center;gap:5px;"><div style="width:14px;height:14px;background:#E1F5EE;border:1.5px solid #1D9E75;border-radius:3px;"></div><span style="font-size:11px;color:var(--color-text-secondary);">Borda — copiada</span></div>
    <div style="display:flex;align-items:center;gap:5px;"><div style="width:14px;height:14px;background:#FBEAF0;border:1.5px solid #993556;border-radius:3px;"></div><span style="font-size:11px;color:var(--color-text-secondary);">Máscara positiva</span></div>
    <div style="display:flex;align-items:center;gap:5px;"><div style="width:14px;height:14px;background:#E6F1FB;border:1.5px solid #185FA5;border-radius:3px;"></div><span style="font-size:11px;color:var(--color-text-secondary);">Máscara negativa</span></div>
  </div>

  <div id="usm-debug" style="background:#E6F1FB;border:0.5px solid #B5D4F4;border-radius:var(--border-radius-md);padding:9px 14px;font-family:var(--font-mono);font-size:11px;color:#0C447C;text-align:center;min-height:36px;line-height:1.7;">
    Passe o mouse sobre um pixel interno do resultado para rastrear o pipeline completo.
  </div>
</div>

<script>
(function(){
  const R=5,C=5,N=25,RAD=1;
  let px=[],bl=[],mk=[],rs=[];
  let k=1.0;

  function tc(g){return g>140?'#111':'#eee';}
  function cmask(v){
    if(Math.abs(v)<2)return'var(--color-background-secondary)';
    if(v>0){const t=Math.min(1,v/80);return`rgb(${Math.round(230+t*20)},${Math.round(210-t*60)},${Math.round(220-t*60)})`;}
    const t=Math.min(1,-v/80);return`rgb(${Math.round(210-t*60)},${Math.round(220-t*40)},${Math.round(230+t*20)})`;
  }

  function gen(){
    px=Array.from({length:N},(_,i)=>{
      const r=Math.floor(i/C),c=i%C;
      let v=(r>=2&&c>=2)?170:70;
      v+=Math.floor(Math.random()*16)-8;
      return Math.max(0,Math.min(255,v));
    });
    calc();
  }

  function calc(){
    bl=new Array(N).fill(0);mk=new Array(N).fill(0);rs=new Array(N).fill(0);
    for(let r=0;r<R;r++){
      for(let c=0;c<C;c++){
        const i=r*C+c;
        if(r>=RAD&&r<R-RAD&&c>=RAD&&c<C-RAD){
          let s=0;
          for(let dr=-RAD;dr<=RAD;dr++)for(let dc=-RAD;dc<=RAD;dc++)s+=px[(r+dr)*C+(c+dc)];
          const b=s/9;const m=px[i]-b;
          bl[i]=b;mk[i]=m;rs[i]=Math.max(0,Math.min(255,Math.round(px[i]+k*m)));
        } else {bl[i]=px[i];mk[i]=0;rs[i]=px[i];}
      }
    }
  }

  function resetDebug(){document.getElementById('usm-debug').innerText='Passe o mouse sobre um pixel interno do resultado para rastrear o pipeline completo.';}

  function highlight(tr,tc2,on){
    const cells=document.getElementById('usm-gf').children;
    for(let i=0;i<N;i++){
      if(!cells[i])continue;
      const p=px[i],r=Math.floor(i/C),c=i%C;
      if(on&&Math.abs(r-tr)<=1&&Math.abs(c-tc2)<=1){
        if(r===tr&&c===tc2){cells[i].style.background='#E6F1FB';cells[i].style.color='#0C447C';cells[i].style.boxShadow='0 0 0 2px #185FA5 inset';}
        else{cells[i].style.background='#FAEEDA';cells[i].style.color='#633806';cells[i].style.boxShadow='0 0 0 2px #BA7517 inset';}
      } else {cells[i].style.background=`rgb(${p},${p},${p})`;cells[i].style.color=tc(p);cells[i].style.boxShadow='none';}
    }
  }

  function render(){
    const gf=document.getElementById('usm-gf'),gb=document.getElementById('usm-gb'),gm=document.getElementById('usm-gm'),gr=document.getElementById('usm-gr');
    gf.innerHTML='';gb.innerHTML='';gm.innerHTML='';gr.innerHTML='';
    document.getElementById('usm-flabel').innerText=`g = f + ${k.toFixed(1)}·m`;

    for(let i=0;i<N;i++){
      const r=Math.floor(i/C),c=i%C;
      const p=px[i],b=bl[i],m=mk[i],res=rs[i];
      const border=(r<RAD||r>=R-RAD||c<RAD||c>=C-RAD);

      const cf=document.createElement('div');cf.className='usm-cell';
      cf.style.cssText=`background:rgb(${p},${p},${p});color:${tc(p)};border:0.5px solid rgba(128,128,128,0.15);`;
      cf.innerText=p;gf.appendChild(cf);

      const cb=document.createElement('div');cb.className='usm-cell';
      if(border){cb.style.cssText=`background:var(--color-background-secondary);color:var(--color-text-secondary);border:0.5px solid var(--color-border-tertiary);`;cb.innerText='—';}
      else{const bv=Math.round(b);cb.style.cssText=`background:rgb(${bv},${bv},${bv});color:${tc(bv)};border:0.5px solid rgba(128,128,128,0.1);`;cb.innerText=b.toFixed(1);}
      gb.appendChild(cb);

      const cm=document.createElement('div');cm.className='usm-cell';
      if(border){cm.style.cssText=`background:var(--color-background-secondary);color:var(--color-text-secondary);border:0.5px solid var(--color-border-tertiary);`;cm.innerText='0';}
      else{const sg=m>=0?'+':'';cm.style.cssText=`background:${cmask(m)};color:#333;border:0.5px solid rgba(128,128,128,0.1);`;cm.innerText=sg+m.toFixed(1);}
      gm.appendChild(cm);

      const crr=document.createElement('div');crr.className='usm-cell';
      if(border){
        crr.style.cssText=`background:#E1F5EE;color:#085041;border:1.5px solid #1D9E75;`;
        crr.onmouseenter=()=>{document.getElementById('usm-debug').innerHTML=`Pixel borda (${r},${c}): copiado sem alteração → <b>${res}</b>`;};
        crr.onmouseleave=resetDebug;
      } else {
        crr.style.cssText=`background:rgb(${res},${res},${res});color:${tc(res)};border:0.5px solid var(--color-border-tertiary);cursor:pointer;`;
        const _r=r,_c=c,_p=p,_b=b,_m=m,_res=res;
        crr.onmouseenter=()=>{
          highlight(_r,_c,true);
          crr.style.transform='scale(1.12)';crr.style.boxShadow='0 0 0 2px #185FA5 inset';
          crr.style.background='#E6F1FB';crr.style.color='#0C447C';
          const sg=_m>=0?'+':'';
          const raw=_p+k*_m;
          document.getElementById('usm-debug').innerHTML=`① f=${_p} &nbsp;→&nbsp; ② f̄=${_b.toFixed(1)} &nbsp;→&nbsp; ③ m=f−f̄=${sg}${_m.toFixed(1)} &nbsp;→&nbsp; ④ g = clip(${_p} + ${k.toFixed(1)}·(${sg}${_m.toFixed(1)})) = clip(${raw.toFixed(1)}) = <b>${_res}</b>`;
        };
        crr.onmouseleave=()=>{
          highlight(_r,_c,false);
          crr.style.transform='scale(1)';crr.style.boxShadow='none';
          crr.style.background=`rgb(${_res},${_res},${_res})`;crr.style.color=tc(_res);
          resetDebug();
        };
      }
      crr.innerText=res;
      gr.appendChild(crr);
    }
  }

  document.getElementById('usm-k').oninput=e=>{k=parseFloat(e.target.value);document.getElementById('usm-klabel').innerText=`k = ${k.toFixed(1)}`;calc();render();};
  document.getElementById('usm-btn').onclick=()=>{gen();render();};
  gen();render();
})();
</script>


""")

In [30]:
%%writefile EP03_10.py
# Código Python

Writing EP03_10.py


In [31]:
TestSuite("EP03_10.py").run()